# Chemical Probes Portal -> probedb

What is in `ChemicalProbesPortal-6_8_2026.json`, which of it the schema in
`database/schema.sql` can hold, and where the field names do not line up.

The rule for this notebook: **count everything, drop nothing silently.** Every
field of the export gets a destination or shows up in the "no home" table at
the end with the number of records it would cost.

Nothing here writes to `staging/`. The staging writer is sketched in
`preprocess.py`; the decisions it needs are the last section of this notebook.

In [1]:
import csv
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 120)


def as_text(value):
    """None, NaN and a blank string all mean 'nothing here'."""
    return "" if value is None or value != value else str(value).strip()


SOURCE = Path("ChemicalProbesPortal-6_8_2026.json")
raw = json.loads(SOURCE.read_text())
probes = raw["probes"]

print(SOURCE, f"{SOURCE.stat().st_size / 1e6:.2f} MB")
print("top-level keys:", list(raw))
print("note:", raw["Note"])
print("probes:", len(probes))

ChemicalProbesPortal-6_8_2026.json 1.33 MB
top-level keys: ['Note', 'probes']
note: Please cite the Chemical Probes Portal
probes: 1247


## 1. Flatten into frames

The export is one object per probe with four nested lists. Every nesting level
becomes its own frame, keyed back to the probe by `probe_ix` so nothing is
orphaned by the flattening itself.

In [2]:
probe = pd.DataFrame(
    [
        {
            "probe_ix": i,
            "name": p["name"],
            "inchikey": p["InChIkey"].strip(),
            "smiles": p["smiles"].strip(),
            "url": p["URL"],
            "published_date": p["published_date"],
            "unsuitable": p["unsuitable"],
            "pains": p["pains"],
            "toxicophore": p["toxicophore"],
            "cansar_id": p["canSAR_ID"],
            "rating_in_cell": p["rating_in_cell"],
            "rating_in_organism": p["rating_in_organism"],
            "rating_count": p["rating_count"],
            "n_chembl": len(p["ChEMBL_ID"]),
            "n_pmid": len(p["PMID"]),
            "n_controls": len(p["control_compounds"]),
            "n_targets": len(p["primary_targets"]),
            "n_invivo": len(p["inVivoValidations"]),
            "n_probe_invitro": len(p["inVitroValidations"]),
        }
        for i, p in enumerate(probes)
    ]
)

target = pd.DataFrame(
    [
        {
            "probe_ix": i,
            "target_ix": j,
            "symbol": t["name"],
            "uniprot_id": t["uniprot_id"],
            "class": t["class"],
            "subclass": t["subClass"],
            "moa": t["moa"],
            "n_invitro": len(t.get("inVitroValidations") or []),
            "n_incell": len(t.get("inCellValidations") or []),
        }
        for i, p in enumerate(probes)
        for j, t in enumerate(p["primary_targets"])
    ]
)

validation = pd.DataFrame(
    [
        {
            "probe_ix": i,
            "target_ix": j,
            "tier": tier,
            "potency": v.get("potency"),
            "potency_value": v.get("potencyValue"),
            "assay_desc": v.get("assayDesc"),
        }
        for i, p in enumerate(probes)
        for j, t in enumerate(p["primary_targets"])
        for tier, key in (("in vitro", "inVitroValidations"), ("in cell", "inCellValidations"))
        for v in (t.get(key) or [])
    ]
)

invivo = pd.DataFrame(
    [
        {"probe_ix": i, "organism": v.get("organism"), "dose": v.get("dose")}
        for i, p in enumerate(probes)
        for v in p["inVivoValidations"]
    ]
)

chembl = pd.DataFrame(
    [
        {"probe_ix": i, "chembl_id": c}
        for i, p in enumerate(probes)
        for c in p["ChEMBL_ID"]
    ]
)

reference = pd.DataFrame(
    [{"probe_ix": i, "ref": r} for i, p in enumerate(probes) for r in p["PMID"]]
)

control = pd.DataFrame(
    [
        {"probe_ix": i, "control_name": c}
        for i, p in enumerate(probes)
        for c in p["control_compounds"]
    ]
)

frames = {
    "probe": probe,
    "target": target,
    "validation": validation,
    "invivo": invivo,
    "chembl": chembl,
    "reference": reference,
    "control": control,
}
pd.DataFrame(
    [{"frame": k, "rows": len(v), "columns": v.shape[1]} for k, v in frames.items()]
)

,frame,rows,columns
0,probe,1247,19
1,target,1372,9
2,validation,3551,6
3,invivo,1265,3
4,chembl,678,2
5,reference,1816,2
6,control,418,2


## 2. Field inventory

Every key in the export, how often it carries something, and how many distinct
values it has. This is the checklist the mapping in section 3 has to cover: 34
paths in all — 2 at the top level, 18 on a probe, 7 on a primary target, 3 on
an in-cell validation, 2 on an in-vitro one, 2 on an in vivo record.

The key union per level is taken over every record, not the first one, because
the levels are not uniform: an in-vitro validation has no `potency` key at all,
and 7 of the 1265 in vivo records have no `dose` key, so `record["dose"]` would
raise on them.

In [3]:
def inventory(records, label):
    """Every key at this level, over the union of records, not the first one.

    `present` counts records carrying the key at all and `filled` counts those
    whose value says something: a key can be absent (in-vitro validations have
    no `potency`, 7 in vivo records have no `dose`) or present and null.
    """
    rows = []
    for k in sorted({k for r in records for k in r}):
        present = [r[k] for r in records if k in r]
        filled = [v for v in present if v not in (None, "", [], {})]
        rows.append(
            {
                "level": label,
                "field": k,
                "records": len(records),
                "present": len(present),
                "filled": len(filled),
                "pct_filled": round(100 * len(filled) / max(len(records), 1), 1),
                "distinct": len({str(v) for v in filled}),
                "example": str(filled[0])[:55] if filled else "",
            }
        )
    return pd.DataFrame(rows)


targets_raw = [t for p in probes for t in p["primary_targets"]]
vitro_raw = [v for p in probes for t in p["primary_targets"]
             for v in (t.get("inVitroValidations") or [])]
cell_raw = [v for p in probes for t in p["primary_targets"]
            for v in (t.get("inCellValidations") or [])]
invivo_raw = [v for p in probes for v in p["inVivoValidations"]]

fields = pd.concat(
    [
        inventory(probes, "probe"),
        inventory(targets_raw, "primary_target"),
        inventory(vitro_raw, "validation (in vitro)"),
        inventory(cell_raw, "validation (in cell)"),
        inventory(invivo_raw, "inVivoValidation"),
    ],
    ignore_index=True,
)
fields

,level,field,records,present,filled,pct_filled,distinct,example
0,probe,ChEMBL_ID,1247,1247,676,54.2,676,['CHEMBL3087498']
1,probe,InChIkey,1247,1247,1213,97.3,1213,IQCKJUKAQJINMK-HUBRGWSESA-N
2,probe,PMID,1247,1247,990,79.4,966,"['www.doi.org/10.1038/ncomms2304', 'http://www.ncbi.nlm"
3,probe,URL,1247,1247,1247,100.0,1247,https://www.chemicalprobes.org/sgc0946
4,probe,canSAR_ID,1247,1247,1214,97.4,1214,828356
5,probe,control_compounds,1247,1247,400,32.1,390,['SGC0649']
6,probe,inVitroValidations,1247,1247,0,0.0,0,
7,probe,inVivoValidations,1247,1247,656,52.6,542,"[{'organism': 'Other', 'dose': '5 mg/mL'}]"
8,probe,name,1247,1247,1247,100.0,1247,SGC0946
9,probe,pains,1247,1247,1247,100.0,2,No


Two things to note before any mapping:

* `inVitroValidations` **on the probe** is empty in all 1247 records. The
  in-vitro numbers live on the primary target, under the same key name. Nothing
  is lost by ignoring the probe-level one.
* a validation record has `potency` (the endpoint) only under
  `inCellValidations`. The in-vitro ones carry `potencyValue` and `assayDesc`
  and no endpoint at all, which is dealt with in section 6.

In [4]:
print("probe-level inVitroValidations, total elements:", probe.n_probe_invitro.sum())
validation.groupby("tier").agg(
    records=("potency", "size"),
    potency_given=("potency", lambda s: s.notna().sum()),
    value_given=("potency_value", lambda s: (s.fillna("").str.strip() != "").sum()),
    desc_given=("assay_desc", lambda s: (s.fillna("").str.strip() != "").sum()),
)

probe-level inVitroValidations, total elements: 0


,records,potency_given,value_given,desc_given
tier,,,,
in cell,1899,1767,1796,1882
in vitro,1652,0,1637,1625


## 3. Name-by-name mapping

Most names differ from the schema's. The column that matters is *why* a portal
name means what it means, because four of them are traps: the portal's `name`,
its `class`, its `organism` and its `PMID` all look like schema columns they are
not.

The table is keyed on the same paths `walk()` produces, and the cell after it
checks that the two sets are equal. That check is the only thing standing
between "every field is accounted for" and a field quietly going missing.

In [5]:
def walk(value, path="", seen=None):
    """Every key path in the export, over every record, not the first one."""
    seen = seen if seen is not None else {}
    if isinstance(value, dict):
        for key, child in value.items():
            seen.setdefault(f"{path}.{key}".lstrip("."), 0)
            seen[f"{path}.{key}".lstrip(".")] += 1
            walk(child, f"{path}.{key}".lstrip("."), seen)
    elif isinstance(value, list):
        for child in value:
            walk(child, f"{path}[]", seen)
    return seen


paths = walk(raw)
print(len(paths), "paths in the export")
pd.DataFrame(sorted(paths.items()), columns=["path", "records carrying it"])

34 paths in the export


,path,records carrying it
0,Note,1
1,probes,1
2,probes[].ChEMBL_ID,1247
3,probes[].InChIkey,1247
4,probes[].PMID,1247
5,probes[].URL,1247
6,probes[].canSAR_ID,1247
7,probes[].control_compounds,1247
8,probes[].inVitroValidations,1247
9,probes[].inVivoValidations,1247


In [6]:
# status: 'mapped' loads today, 'decision' can only load once a decision is
# taken, 'no column' has nowhere to go, 'not data' is not a fact about anything
MAPPING = [
    ("Note", "-", "not data",
     "'Please cite the Chemical Probes Portal', a citation notice"),
    ("probes", "-", "not data", "the container"),
    ("probes[].name", "compound.name", "mapped",
     "the probe's trivial name and the only name given, so compound.name gets it "
     "even though 18 probes are named by a bare ChEMBL id and 10 need stripping"),
    ("probes[].InChIkey", "compound.inchikey", "mapped",
     "same identifier, portal lowercases the k; standard 14-10-1 key, the PK "
     "everywhere, and it agrees with the SMILES in 1190 of 1191 records"),
    ("probes[].smiles", "compound.smiles", "mapped", "as given, 29 not canonical (D16)"),
    ("probes[].ChEMBL_ID", "chembl.chembl_id", "mapped",
     "list; chembl is keyed on the id so two ids for one compound is two rows"),
    ("probes[].primary_targets[].uniprot_id", "uniprot.uniprot_id", "mapped",
     "accession, one per portal target entry"),
    ("probes[].primary_targets[].name", "uniprot.hgnc + target.name", "mapped",
     "the portal calls it 'name' but the values are HGNC gene symbols (BRD4, "
     "MDM2), so the symbol goes to uniprot.hgnc. target.name wants a protein "
     "name, which the export does not carry, so the symbol is reused as the "
     "label and whichever source loads first owns it"),
    ("probes[].primary_targets[].moa", "bioactivity.moa", "mapped",
     "same concept, mixed case and 35 spellings, normalised in section 6"),
    ("probes[].primary_targets[].inCellValidations[].potency",
     "bioactivity.bioactivity_type", "mapped",
     "'potency' here is the endpoint label (IC50, DC50, Dmax), not a number; the "
     "number is in potencyValue"),
    ("probes[].primary_targets[].inCellValidations[].potencyValue",
     "bioactivity.relation/value/unit", "mapped",
     "free text holding operator, number, unit and often several measurements"),
    ("probes[].primary_targets[].inCellValidations[].assayDesc",
     "bioactivity.assay_description", "mapped",
     "same concept; longer than the declared VARCHAR(255) in 84 records (D11)"),
    ("probes[].primary_targets[].inCellValidations",
     "bioactivity.assay_type = 'cell'", "mapped", "the tier is the assay type"),
    ("probes[].primary_targets[].inVitroValidations[].potencyValue",
     "bioactivity.relation/value/unit", "mapped", "as above, and no endpoint (D6)"),
    ("probes[].primary_targets[].inVitroValidations[].assayDesc",
     "bioactivity.assay_description", "mapped", "as above"),
    ("probes[].primary_targets[].inVitroValidations",
     "bioactivity.assay_type = 'biochemical'", "mapped",
     "the portal's 'in vitro' tier means cell-free, which is what the schema's "
     "'biochemical' means; 'in vitro' in the wider literature includes cell "
     "assays, so the tier name cannot be copied across as a value"),
    ("probes[].primary_targets", "target.type = 'protein'", "mapped",
     "every portal target entry is exactly one accession, so it is a one-member "
     "target. class/subClass are NOT this column"),
    ("probes[].URL", "bioactivity_source.xref_id + source_xref", "mapped",
     "the portal record for the probe is the provenance of every number under "
     "it; the prefix plus the full path rebuilds the URL"),
    ("probes[].inVitroValidations", "-", "not data",
     "the probe-level list, empty in all 1247 records; the in-vitro numbers live "
     "under the primary target, under the same key name"),
    ("probes[].inVivoValidations", "in_vivo table", "decision",
     "the container for the in vivo records, see D5"),
    ("probes[].inVivoValidations[].organism", "in_vivo.organism", "decision",
     "TRAP: uniprot.species is the species of the target protein, this is the "
     "animal the probe was dosed in. Writing it to species would be wrong, and "
     "concentration/_unit is wrong too: mg/kg is a dose, not a concentration"),
    ("probes[].inVivoValidations[].dose", "in_vivo.dose_value/_unit", "decision",
     "a dose administered, not a potency; a bioactivity row needs a target_id "
     "which an in vivo record does not have. Absent as a key in 7 records"),
    ("probes[].PMID", "compound_annotation", "decision",
     "named after PubMed IDs but holds 1019 DOI urls, 599 PubMed urls and 198 "
     "publisher urls, and it is a probe-level list, not per measurement, so it "
     "cannot be attached to one number without inventing a link. See D4"),
    ("probes[].primary_targets[].class", "target_annotation", "no column",
     "TRAP: target.type is a CHECK'd vocabulary about composition "
     "(protein/complex/ppi/family). 'Kinase', 'Epigenetic', 'GPCR' are a protein "
     "family taxonomy, a different axis, and not a property of the accession"),
    ("probes[].primary_targets[].subClass", "target_annotation", "no column",
     "second level of the same taxonomy"),
    ("probes[].rating_in_cell", "compound_annotation", "no column",
     "portal expert review score, 0-4, for cell data"),
    ("probes[].rating_in_organism", "compound_annotation", "no column",
     "same score for whole-organism data"),
    ("probes[].rating_count", "compound_annotation", "no column",
     "how many reviewers contributed the scores"),
    ("probes[].unsuitable", "compound_annotation", "no column",
     "portal verdict that the compound should not be used as a probe; the 260 "
     "Yes records are exactly the 260 with no targets, so without this flag "
     "they look like missing data rather than a judgement"),
    ("probes[].pains", "compound_annotation", "no column",
     "pan-assay interference substructure alert, compound level"),
    ("probes[].toxicophore", "compound_annotation", "no column",
     "toxicophore substructure alert, compound level"),
    ("probes[].canSAR_ID", "compound_annotation", "no column",
     "canSAR compound id, an xref like chembl_id"),
    ("probes[].published_date", "compound_annotation", "no column",
     "date the portal record was published"),
    ("probes[].control_compounds", "compound_annotation", "no column",
     "names of negative-control compounds; names only, no structure, and "
     "compound is keyed on InChIKey, so they cannot be compound rows"),
]
mapping = pd.DataFrame(MAPPING, columns=["path", "destination", "status", "why"])
mapping

,path,destination,status,why
0,Note,-,not data,"'Please cite the Chemical Probes Portal', a citation notice"
1,probes,-,not data,the container
2,probes[].name,compound.name,mapped,"the probe's trivial name and the only name given, so compound.name gets it even though..."
3,probes[].InChIkey,compound.inchikey,mapped,"same identifier, portal lowercases the k; standard 14-10-1 key, the PK everywhere, and..."
4,probes[].smiles,compound.smiles,mapped,"as given, 29 not canonical (D16)"
5,probes[].ChEMBL_ID,chembl.chembl_id,mapped,list; chembl is keyed on the id so two ids for one compound is two rows
6,probes[].primary_targets[].uniprot_id,uniprot.uniprot_id,mapped,"accession, one per portal target entry"
7,probes[].primary_targets[].name,uniprot.hgnc + target.name,mapped,"the portal calls it 'name' but the values are HGNC gene symbols (BRD4, MDM2), so the s..."
8,probes[].primary_targets[].moa,bioactivity.moa,mapped,"same concept, mixed case and 35 spellings, normalised in section 6"
9,probes[].primary_targets[].inCellValidations[].potency,bioactivity.bioactivity_type,mapped,"'potency' here is the endpoint label (IC50, DC50, Dmax), not a number; the number is i..."


In [7]:
# the completeness check: the mapping and the export have to name the same paths
missing = sorted(set(paths) - set(mapping.path))
invented = sorted(set(mapping.path) - set(paths))
print("paths in the export not in the mapping:", missing or "none")
print("paths in the mapping not in the export:", invented or "none")
assert not missing and not invented, "the mapping no longer covers the export"
print()
print(mapping.status.value_counts().to_string())

paths in the export not in the mapping: none
paths in the mapping not in the export: none

status
mapped       16
no column    11
decision      4
not data      3


## 4. Compounds

`compound.inchikey` is the primary key and the foreign key on every bioactivity
row, so a probe without one cannot be loaded at all under the current schema.

In [8]:
INCHIKEY = re.compile(r"^[A-Z]{14}-[A-Z]{10}-[A-Z]$")
CHEMBL = re.compile(r"^CHEMBL[0-9]+$")

has_key = probe.inchikey != ""
print("probes:", len(probe))
print("with an InChIKey:", has_key.sum(), " without:", (~has_key).sum())
print("malformed InChIKeys:", (~probe.loc[has_key, "inchikey"].str.match(INCHIKEY)).sum())
print("duplicate InChIKeys:", probe.loc[has_key, "inchikey"].duplicated().sum())
print("with a SMILES:", (probe.smiles != "").sum())
print("InChIKey but no SMILES:", (has_key & (probe.smiles == "")).sum())
print("SMILES but no InChIKey:", ((~has_key) & (probe.smiles != "")).sum())
print("malformed ChEMBL ids:", (~chembl.chembl_id.str.match(CHEMBL)).sum())
print("ChEMBL ids on >1 probe:", chembl.groupby("chembl_id").probe_ix.nunique().gt(1).sum())
print("probes with 2 ChEMBL ids:", (probe.n_chembl == 2).sum())

probes: 1247
with an InChIKey: 1213  without: 34
malformed InChIKeys: 0
duplicate InChIKeys: 0
with a SMILES: 1191
InChIKey but no SMILES: 22
SMILES but no InChIKey: 0
malformed ChEMBL ids: 0
ChEMBL ids on >1 probe: 0
probes with 2 ChEMBL ids: 2


### The 34 probes with no InChIKey

None of them has a SMILES either, so the key cannot be recomputed offline. They
are recent entries the portal has not resolved to a structure yet. Between them
they carry 41 target entries and the validations below, all of which the loader
would have to drop.

In [9]:
no_key = probe[~has_key]
lost = validation.merge(no_key[["probe_ix"]], on="probe_ix")
print("probes with no InChIKey:", len(no_key))
print("their target entries:", target.merge(no_key[['probe_ix']], on='probe_ix').shape[0])
print("their validation records:", len(lost))
print("their in vivo records:", invivo.merge(no_key[['probe_ix']], on='probe_ix').shape[0])
print("their ChEMBL ids:", chembl.merge(no_key[['probe_ix']], on='probe_ix').shape[0])
no_key[["name", "url", "n_targets", "n_chembl", "unsuitable"]].head(40)

probes with no InChIKey: 34
their target entries: 41
their validation records: 119
their in vivo records: 58
their ChEMBL ids: 1


,name,url,n_targets,n_chembl,unsuitable
1042,ITACITINIB,https://www.chemicalprobes.org/itacitinib,1,1,No
1095,C3TD879,https://www.chemicalprobes.org/c3td879,1,0,No
1104,ART5537,https://www.chemicalprobes.org/art5537,1,0,No
1108,LC-04-45,https://www.chemicalprobes.org/lc-04-45,1,0,No
1135,TO-1187,https://www.chemicalprobes.org/to-1187,1,0,No
1161,D16-M1P2,https://www.chemicalprobes.org/d16-m1p2,1,0,No
1174,BMS-986458,https://www.chemicalprobes.org/bms-986458,1,0,No
1179,SGC-CK1γ-1,https://www.chemicalprobes.org/cpd14,3,0,No
1190,FGFR3-IN-10s,https://www.chemicalprobes.org/fgfr3-in-10s,1,0,No
1191,LS-170,https://www.chemicalprobes.org/ls-170,1,0,No


### Probes with no target

260 probes list no primary target, so they contribute a compound row and
nothing else. They are exactly the 260 marked `unsuitable = Yes`: the portal
strips target and validation data from compounds it has judged unsuitable. The
two facts are the same fact, which is why dropping `unsuitable` would leave 260
compounds that look like an incomplete download.

In [10]:
no_target = probe[probe.n_targets == 0]
print("probes with 0 primary targets:", len(no_target))
print("of those, unsuitable=Yes:", (no_target.unsuitable == "Yes").sum())
print("probes with unsuitable=Yes overall:", (probe.unsuitable == "Yes").sum())
print("their in vivo records:", no_target.n_invivo.sum())
print("their references:", no_target.n_pmid.sum())
pd.crosstab(probe.n_targets.clip(upper=5), probe.unsuitable)

probes with 0 primary targets: 260
of those, unsuitable=Yes: 260
probes with unsuitable=Yes overall: 260
their in vivo records: 0
their references: 8


unsuitable,No,Yes
n_targets,,
0,0,260
1,730,0
2,169,0
3,54,0
4,28,0
5,6,0


### Does the SMILES agree with the InChIKey?

Both go into `compound`, and the InChIKey is the primary key, so if the two
disagree the row is self-contradictory and every lookup by structure is
unreliable. The key is a hash of the structure, so this is checkable: recompute
it from the SMILES and compare.

Needs RDKit (`uv pip install rdkit`). The rest of the notebook does not.

In [11]:
try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem.inchi import MolToInchiKey

    RDLogger.DisableLog("rdApp.*")
except ImportError:
    Chem = None
    print("rdkit not installed, skipping the structure check")

if Chem is not None:
    structures = []
    for row in probe[has_key & (probe.smiles != "")].itertuples():
        mol = Chem.MolFromSmiles(row.smiles)
        structures.append({
            "name": row.name,
            "given": row.inchikey,
            "computed": MolToInchiKey(mol) if mol else None,
            "canonical": Chem.MolToSmiles(mol) if mol else None,
            "smiles": row.smiles,
        })
    structures = pd.DataFrame(structures)
    structures["agrees"] = structures.given == structures.computed
    structures["same_skeleton"] = (
        structures.given.str.split("-").str[0] == structures.computed.str.split("-").str[0]
    )
    structures["is_canonical"] = structures.smiles == structures.canonical

    print("probes with both an InChIKey and a SMILES:", len(structures))
    print("  SMILES RDKit cannot parse:", structures.canonical.isna().sum())
    print("  recomputed key == given key:", structures.agrees.sum())
    print("  same skeleton, different stereo block:",
          (structures.same_skeleton & ~structures.agrees).sum())
    print("  different skeleton:", (~structures.same_skeleton).sum())
    print("  SMILES already in RDKit canonical form:", structures.is_canonical.sum())
    display(structures.loc[~structures.agrees, ["name", "given", "computed", "smiles"]])

probes with both an InChIKey and a SMILES: 1191
  SMILES RDKit cannot parse: 0
  recomputed key == given key: 1190
  same skeleton, different stereo block: 1
  different skeleton: 0
  SMILES already in RDKit canonical form: 1162


,name,given,computed,smiles
200,NSC117907,JPOAXWSMFOLMQH-UWWJMHSNSA-N,JPOAXWSMFOLMQH-UHFFFAOYSA-N,O=C(O)c1cc(N=C2C=CC(=C(c3ccccc3)c3ccc(Nc4ccc(Cl)c(C(=O)O)c4)cc3)C=C2)ccc1Cl


1190 of 1191 agree exactly, so the pairing can be trusted and there is no
reason to recompute keys on load.

The one exception is `NSC117907`: the export gives
`JPOAXWSMFOLMQH-UWWJMHSNSA-N` but its SMILES hashes to
`JPOAXWSMFOLMQH-UHFFFAOYSA-N`. Same skeleton, and `UHFFFAOYSA` is the hash of
an empty stereo layer, so the SMILES carries no configuration at all while the
key claims one. RDKit finds no stereocentre in it: the geometry is on the
quinoid `C=N`, which the SMILES writes without it. The key is the more specific
of the two, and it is the primary key, so it wins; the SMILES is the lossy
column here.

### 29 SMILES are not canonical

All 29 are the same molecule after re-canonicalising, so this is only how the
string was written, by whichever toolkit the curator used. It matters for one
thing: comparing `compound.smiles` between two sources as strings. The
InChIKey is the join key everywhere, so nothing in the schema depends on it.
Decision D16.

In [12]:
if Chem is not None:
    not_canonical = structures[~structures.is_canonical & structures.canonical.notna()]
    same_molecule = sum(
        Chem.MolToSmiles(Chem.MolFromSmiles(r.smiles))
        == Chem.MolToSmiles(Chem.MolFromSmiles(r.canonical))
        for r in not_canonical.itertuples()
    )
    print(f"same molecule after re-canonicalising: {same_molecule}/{len(not_canonical)}")
    display(not_canonical[["name", "smiles", "canonical"]].head(6))

same molecule after re-canonicalising: 29/29


,name,smiles,canonical
5,KU-60019,C[C@H]1CN(CC(=O)Nc2ccc3c(c2)Cc2cccc(-c4cc(=O)cc(N5CCOCC5)o4)c2S3)C[C@@H](C)O1,C[C@@H]1CN(CC(=O)Nc2ccc3c(c2)Cc2cccc(-c4cc(=O)cc(N5CCOCC5)o4)c2S3)C[C@H](C)O1
90,SL0101,CC(=O)O[C@H]1[C@H](C)O[C@@H](Oc2c(-c3ccc(O)cc3)oc3cc(O)cc(O)c3c2=O)[C@H](O)[C@@H]1OC(C)=O,CC(=O)O[C@@H]1[C@@H](OC(C)=O)[C@@H](O)[C@H](Oc2c(-c3ccc(O)cc3)oc3cc(O)cc(O)c3c2=O)O[C@...
107,U18666A,CCN(CC)CCO[C@H]1CC[C@@]2(C)C(=CC[C@H]3[C@@H]4CCC(=O)[C@@]4(C)CC[C@@H]32)C1.Cl,CCN(CC)CCO[C@H]1CC[C@@]2(C)C(=CC[C@@H]3[C@@H]2CC[C@]2(C)C(=O)CC[C@@H]32)C1.Cl
137,AT-101,O[C@H]1CO[C@@H]2[C@H](O)CO[C@H]12,O[C@@H]1CO[C@H]2[C@@H]1OC[C@@H]2O
167,Brusatol,COC(=O)[C@@]12OC[C@@]34[C@H]1[C@@H](OC(=O)C=C(C)C)C(=O)O[C@@H]3C[C@H]1C(C)=C(O)C(=O)C[...,COC(=O)[C@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)[C@@]1(C)CC(=O)C(O)=C(C)[C@@H]1C[C@H]3OC(...
182,4-PBHA,Cl.N#C[C@@H]1CS[C@H]2C[C@](N)(C3CCCC3)C(=O)N21,Cl.N#C[C@@H]1CS[C@H]2C[C@](N)(C3CCCC3)C(=O)N12


### Two records for one compound

No InChIKey repeats, but four pairs share a 14-character skeleton and differ
only in the stereo block. Three of the four are genuine stereoisomer pairs and
must stay separate: `Crizotinib` / `S-crizotinib` and one CHEMBL-named pair are
the enantiomer hashes `GFCCVEGC` / `LBPRGKRZ`, and another pair is an E/Z
pair. The fourth is not a pair of isomers at all.

In [13]:
skeleton = probe[has_key].assign(
    skeleton=probe.loc[has_key, "inchikey"].str.split("-").str[0]
)
twins = skeleton[skeleton.skeleton.duplicated(keep=False)].sort_values("skeleton")
print("14-character skeletons shared by more than one probe:", twins.skeleton.nunique())
twins[["name", "inchikey", "n_targets", "unsuitable", "url"]]

14-character skeletons shared by more than one probe: 4


,name,inchikey,n_targets,unsuitable,url
220,CHEMBL2179989,GNFSYBNDPOBXLJ-PLNGDYQASA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/chembl2179989
237,CHEMBL2179988,GNFSYBNDPOBXLJ-SNAWJCMRSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/chembl2179988
221,CHEMBL3092538,HWTVYWVFOWWESR-GFCCVEGCSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/chembl3092538
224,CHEMBL3092539,HWTVYWVFOWWESR-LBPRGKRZSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/chembl3092539
37,Crizotinib,KTEIFNKAUNYNJU-GFCCVEGCSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/crizotinib
206,S-crizotinib,KTEIFNKAUNYNJU-LBPRGKRZSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/s-crizotinib
45,Intedanib,XZXHXSATPCNXJR-ZIADKAODSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/intedanib
94,Ninetedanib,XZXHXSATPCNXJR-UHFFFAOYSA-N,0,Yes,https://www.chemicalprobes.org/unsuitables/ninetedanib


`Intedanib` and `Ninetedanib` are two spellings of nintedanib, entered twice,
with different ChEMBL ids and different canSAR ids, and identical structures
once stereochemistry is ignored — one record recorded the configuration and the
other did not. As two InChIKeys they become two `compound` rows for one
compound. Both are `unsuitable` with no targets, so they cost two compound rows
and nothing else, but they are a duplicate the portal is carrying, not
something this preprocessing introduces. Decision D17.

## 5. Targets

The portal's target entry is a gene symbol plus one accession. Symbol and
accession are in strict 1:1 agreement across all 1372 entries, and no entry
packs two proteins into one string, so every entry is a one-member `protein`
target and no complex or family has to be inferred.

In [14]:
print("target entries:", len(target))
print("distinct symbols:", target.symbol.nunique(), " distinct accessions:", target.uniprot_id.nunique())
print("symbols mapping to >1 accession:", target.groupby("symbol").uniprot_id.nunique().gt(1).sum())
print("accessions mapping to >1 symbol:", target.groupby("uniprot_id").symbol.nunique().gt(1).sum())
print("entries whose symbol contains a separator:",
      target.symbol.str.contains(r"[,;/|+]| and ").sum())
UNIPROT = re.compile(r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2})$")
print("non-canonical accessions:", (~target.uniprot_id.str.match(UNIPROT)).sum())
print("same accession twice under one probe:",
      target.groupby(["probe_ix", "uniprot_id"]).size().gt(1).sum())
print("\nprobes per target count:")
print(probe.n_targets.value_counts().sort_index())

target entries: 1372
distinct symbols: 656  distinct accessions: 656
symbols mapping to >1 accession: 0
accessions mapping to >1 symbol: 0
entries whose symbol contains a separator: 0
non-canonical accessions: 0
same accession twice under one probe: 0

probes per target count:
n_targets
0    260
1    730
2    169
3     54
4     28
5      6
Name: count, dtype: int64


`species` and `entrez_gene` on `uniprot` have no source in the export. The
portal is human-target focused but a handful of assay descriptions name
parasite and rodent orthologues, so filling `species` with 'Homo sapiens'
would be a guess. Left empty; it is a UniProt lookup, not a preprocessing step.

### class / subClass

A protein family taxonomy with 56 classes and 329 subclasses, free text, some
of it typos ('Programmed Cell Deatch', 'Transcriptional Factor' next to
'Transcription factor'). It is not `target.type`, and there is nowhere else for
it to go.

It is also not a property of the accession, which rules out the obvious home.
69 of 656 accessions are filed under more than one class and 126 under more
than one subclass, because the value belongs to the portal's (probe, target)
curation row: P51812 is both 'Kinase' and 'Protein kinase', Q13822 carries
four different subclasses. Keying it on `uniprot_id` would make one of them
win silently.

In [15]:
blank_subclass = target.subclass.map(as_text).eq("")
print(target["class"].nunique(), "classes,",
      target.loc[~blank_subclass, "subclass"].nunique(), "subclasses,",
      blank_subclass.sum(), "entries with no subclass")
# the taxonomy is not a property of the protein: the same accession is filed
# under different classes by different probe records
print("accessions with >1 class:   ",
      target.groupby("uniprot_id")["class"].nunique().gt(1).sum(), "of", target.uniprot_id.nunique())
print("accessions with >1 subclass:",
      target[~blank_subclass].groupby("uniprot_id").subclass.nunique().gt(1).sum())
for accession in ["P51812", "Q00987", "Q13822"]:
    values = sorted(set(target.loc[target.uniprot_id == accession, "class"])
                    | set(target.loc[target.uniprot_id == accession, "subclass"]))
    print(f"   {accession}: {values}")
target["class"].value_counts().head(20)

56 classes, 329 subclasses, 51 entries with no subclass
accessions with >1 class:    69 of 656
accessions with >1 subclass: 126
   P51812: ['AGC', 'Kinase', 'Protein kinase']
   Q00987: ['E3 ubiquitin ligase', 'Enzyme', 'Ligase', 'Other post-translational modification', 'Post translational modifications', 'Ubiquitin Ligase']
   Q13822: ['Autotaxin', 'Enzyme', 'Other', 'Phosphodiesterase', 'nucleotide pyrophosphatase/phosphodiesterase; lysophospholipase ', 'pyrophosphatase/phosphodiesterase']


class
Kinase                                   520
Epigenetic                               284
Enzyme                                   156
GPCR                                     114
Other post-translational modification     62
Protein kinase                            40
Ion Channel                               33
Other                                     31
NHR                                       27
Transcription factor                      20
Transporter                               15
Lipid metabolism                           6
Protein metabolism                         5
Lipid kinase                               5
Structural protein                         4
Nucleic acid metabolism                    4
Cytohesin                                  3
Nuclear Hormone Receptor                   2
Toll-Like Receptor                         2
Translation Termination Factor             2
Name: count, dtype: int64

### MoA normalisation

`moa` is part of the `bioactivity_group` unique key, so spelling variants split
a group in two. Casing and stray whitespace account for most of the 35 raw
spellings, which collapse to 30; two are fixed by hand. The 12 entries naming
more than one mechanism stay verbatim (lowercased), because collapsing
'Degrader, Antagonist' to one word would invent a claim.

In [16]:
def normalise_moa(value):
    return re.sub(r"\s+", " ", as_text(value)).lower()


MOA_FIX = {
    "antagoinist, degrader": "antagonist, degrader",   # typo in the export
    "molecular glues": "molecular glue",               # plural of a single mechanism
}
target["moa_norm"] = target.moa.map(normalise_moa).replace(MOA_FIX)
moa_map = (
    target.groupby(["moa", "moa_norm"]).size().rename("entries").reset_index()
    .sort_values("entries", ascending=False)
)
print("raw spellings:", target.moa.nunique(), "-> normalised:", target.moa_norm.nunique())
print("entries whose moa names more than one mechanism:",
      target.moa_norm.str.contains(r",|/| and ").sum())
moa_map

raw spellings: 35 -> normalised: 30
entries whose moa names more than one mechanism: 12


,moa,moa_norm,entries
17,Inhibitor,inhibitor,928
6,Antagonist,antagonist,125
13,Degrader (PROTAC),degrader (protac),123
3,Agonist,agonist,57
11,Covalent Inhibitor,covalent inhibitor,55
22,Modulator,modulator,17
24,Molecular Glues,molecular glue,10
8,Bivalent inhibitor,bivalent inhibitor,8
19,Inverse Agonist,inverse agonist,7
2,Activator,activator,7


## 6. Endpoints

`potency` is the endpoint label and only in-cell validations have it. The
vocabulary needs collapsing ('IC 50', 'IC50 ', 'IC50' are one endpoint) and 15
records name several endpoints at once, one per number in `potencyValue`.

In [17]:
ENDPOINT_FIX = {
    "ic 50": "IC50", "ec 50": "EC50", "dc 50": "DC50", "gi 50": "GI50",
    "gi50*": "GI50", "k 0.5": "K0.5", "kd apparent": "Kd(app)", "dc50 @ 16h": "DC50",
    "inh": "% inhibition", "inhibition": "% inhibition", "% in": "% inhibition",
    "% ac": "% activation", "activity": "activity", "residual activity": "residual activity",
    "pkbb": "pKb", "not done": None, "ratio": "ratio", "dmax": "Dmax", "delta tm": "ΔTm",
}
CANON = {e.lower(): e for e in
         ["IC50", "EC50", "DC50", "GI50", "Ki", "Kd", "Dmax", "ΔTm", "pIC50", "pEC50",
          "pGI50", "pKb", "IC30", "K0.5", "Kd(app)", "MEC1.5", "MEC2.0", "Emax"]}


def normalise_endpoint(value):
    """One endpoint label -> canonical spelling. None for 'not done' and blanks."""
    v = re.sub(r"\s+", " ", as_text(value))
    if not v or v.lower() in ("none", "nan"):
        return None
    low = v.lower()
    if low in ENDPOINT_FIX:
        return ENDPOINT_FIX[low]
    if low in CANON:
        return CANON[low]
    return re.sub(r"\b(IC|EC|DC|GI) ?50\b", r"\g<1>50", v, flags=re.I)


def split_endpoints(value):
    """'DC50, Dmax, IC50' -> ['DC50', 'Dmax', 'IC50']"""
    parts = [normalise_endpoint(p) for p in re.split(r"[,;]", as_text(value))]
    return [p for p in parts if p]


cell_potency = validation[validation.tier == "in cell"].potency
endpoints = (
    pd.DataFrame({"raw": cell_potency, "norm": cell_potency.map(normalise_endpoint)})
    .value_counts(dropna=False).rename("records").reset_index()
)
print("raw labels:", cell_potency.nunique(), "-> normalised:",
      cell_potency.map(normalise_endpoint).nunique())
print("records naming >1 endpoint:", cell_potency.map(lambda v: len(split_endpoints(v)) > 1).sum())
endpoints.head(60)

raw labels: 43 -> normalised: 32
records naming >1 endpoint: 15


,raw,norm,records
0,IC50,IC50,872
1,EC50,EC50,318
2,NaN,NaN,132
3,DC50,DC50,124
4,IC 50,IC50,105
5,Dmax,Dmax,79
6,Activity,activity,46
7,EC 50,EC50,44
8,INH,% inhibition,34
9,GI50,GI50,31


### In-vitro records have no endpoint

The export simply does not carry one for the 1652 in-vitro validations. 65 of
them name it inside `assayDesc` ('SPR; IC50 = 0.3 nM in enzymatic assay'). The
rest would be `bioactivity_type = NULL`, with the raw description kept, which
is honest but makes those rows hard to compare. Decision D6.

In [18]:
DESC_ENDPOINT = re.compile(
    r"\b(pIC ?50|pEC ?50|pK ?[DdIi]|IC ?50|EC ?50|K ?D\b|K ?[di]\b|GI ?50|DC ?50|"
    r"Dmax|Kinact|ΔTm)\b")
vitro = validation[validation.tier == "in vitro"]
found = vitro.assay_desc.fillna("").str.extract(DESC_ENDPOINT)[0]
print("in-vitro records:", len(vitro))
print("endpoint recoverable from assayDesc:", found.notna().sum())
found.dropna().str.replace(" ", "", regex=False).str.upper().value_counts()

in-vitro records: 1652
endpoint recoverable from assayDesc: 65


0
IC50      35
KD        13
KI        11
KINACT     4
PKD        1
PIC50      1
Name: count, dtype: int64

### Assay technique

`assayDesc` names the technique often enough to tell a binding assay from an
enzymatic one, which is the distinction the template's `assay_type` makes
('binding' for a Kd, 'biochemical' for an enzyme IC50). Counts for the keyword
rule are below; whether to use it is decision D7.

In [19]:
BINDING = re.compile(r"\b(?:SPR|ITC|BROMOscan|thermal shift|DSF|MST|NanoBRET|"
                     r"fluorescence polarization|radioligand|scintillation)\b", re.I)
ENZYMATIC = re.compile(r"\b(?:enzymatic|biochemical|TR-?FRET|HTRF|AlphaScreen|alpha screen|"
                       r"methyltransferase|kinase activity)\b", re.I)
tech = pd.DataFrame({
    "binding_kw": vitro.assay_desc.fillna("").str.contains(BINDING),
    "enzymatic_kw": vitro.assay_desc.fillna("").str.contains(ENZYMATIC),
})
print(tech.value_counts().rename("in-vitro records"))

binding_kw  enzymatic_kw
False       False           793
            True            480
True        False           367
            True             12
Name: in-vitro records, dtype: int64


## 7. Parsing potencyValue

One free-text field has to become `relation`, `value` and `unit`. Of the 3551
records, 3316 hold exactly one number, 110 hold several, and 125 hold none.
Around the numbers sit operators, standard deviations, ranges, assay
concentrations, and labels saying which domain, species, mutant or cell line
each number belongs to.

The parser splits a record into fragments, takes one number per fragment and
never invents one. Where it cannot read a value reliably it sets `quarantine`
rather than guessing, so those rows can be curated by hand instead of loaded.

This is the second version. An adversarial pass over all 3551 records broke the
first one in twelve places; the fixes are marked in the code and listed under
"what the first version got wrong" below.

In [20]:
NUM = r"\d+(?:\.\d+)?"

# rate constants and reciprocal units come first, they contain the shorter
# units. the ambiguous single letters M, K and h must not sit inside a word or
# follow '(', or 'T790M' is molar, '(K)' a temperature and '(H)' an hour.
UNIT_ALT = [
    (r"M\s*[−–-]\s*1\s*[·.]?\s*[sS]\s*[−–-]\s*1"
     r"|[sS]\s*[−–-]\s*1\s*[·.]?\s*M\s*[−–-]\s*1"
     r"|/\(mol/L\)\s*s|per\s*s/[uµμ]mol/L|per\s*M\s*per\s*sec", "M-1s-1"),
    (r"min\s*[−–-]\s*1", "min-1"),
    (r"nmol/L", "nM"), (r"[uµμ]mol/L", "uM"), (r"mmol/L", "mM"), (r"mol/L", "M"),
    (r"pM\b", "pM"), (r"nM\b", "nM"), (r"[uµμ]M\b", "uM"), (r"mM\b", "mM"),
    (r"(?<![A-Za-z0-9(])M\b", "M"),
    (r"%|Percent\s*of\s*Control", "%"),
    (r"°\s*C|ºC|Celciuys|Celsius|degrees|Kelvin|(?<![A-Za-z0-9(])K\b", "degC"),
    (r"fold|(?<=\d)x\b", "fold"),
    (r"(?<![A-Za-z0-9(])h\b|hours?\b", "h"),
    (r"(?<![A-Za-z0-9(])min\b", "min"),
]
UNIT_RE = re.compile("|".join(f"(?P<u{i}>{p})" for i, (p, _) in enumerate(UNIT_ALT)), re.I)
UNIT_CANON = {f"u{i}": c for i, (_, c) in enumerate(UNIT_ALT)}
ANY_UNIT = "|".join(p for p, _ in UNIT_ALT)
MOLAR = {"pM", "nM", "uM", "mM", "M"}

RELATION = {"<": "<", ">": ">", "~": "~", "=": "=", "≤": "<=", "≥": ">=",
            "<=": "<=", ">=": ">=", "≈": "~"}
WORD_RELATION = [                       # a censored value must not become exact
    (r"\b(?:less than|below|under|up to|at most|fewer than)\b", "<"),
    (r"\b(?:greater than|more than|above|over|at least)\b", ">"),
    (r"\b(?:about|around|approximately|approx\.?|circa|ca\.)\b", "~"),
]
NOT_A_VALUE = {"", "none", "na", "n/a", "nd", "not determined", "not done",
               "-", "not available"}
P_SCALE = re.compile(r"^p[A-Z]", re.I)
PERCENT_ENDPOINT = re.compile(r"^(?:%|dmax|emax)", re.I)
# the concentration an assay ran at, not the measurement: '6.1 nM @ 10 uM ATP'
AT_CONCENTRATION = re.compile(rf"(?:@|\bat\b)\s*(?P<value>{NUM})\s*(?P<unit>{ANY_UNIT})", re.I)
EXPONENT = re.compile(r"[xX×*]\s*10\s*[\^*]?\s*[-−–+]?\d+|[eE][-+]\d+")
SLASHED_NUMBERS = re.compile(rf"(?<![A-Za-z0-9]){NUM}\s*/\s*{NUM}(?![A-Za-z0-9])")
FRAGMENT = re.compile(
    rf"""(?P<rel>[<>~≤≥≈]=?|=)?\s*
         (?<![A-Za-z0-9.])(?P<lo>{NUM})
         (?:\s*(?:±|\+/-|\+-)\s*(?P<err>{NUM}))?
         (?:\s*(?:{ANY_UNIT})?\s*(?:[-–—]|\bto\b)\s*(?P<hi>{NUM})(?![0-9.]))?""",
    re.X | re.I,
)


def repair(text):
    """Typography that has to be fixed before the string is split up."""
    text = re.sub(r"(?<=\d)\.\s+(?=\d)", ".", text)          # '0. 69'  -> '0.69'
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text)          # '1,100'  -> '1100'
    text = re.sub(r"(?<=\d),(?=\d(?!\d{2}))", ".", text)     # '6,5 nM' -> '6.5 nM'
    text = re.sub(r"(?<=\d)\s*n\s+M\b", " nM", text)         # '85n M'  -> '85 nM'
    for pattern, symbol in WORD_RELATION:
        text = re.sub(pattern, symbol, text, flags=re.I)
    return text


def canonical_unit(text):
    match = UNIT_RE.search(as_text(text) or "")
    return UNIT_CANON[match.lastgroup] if match else None


def split_value_fragments(text):
    """A repaired potencyValue -> its value-bearing fragments."""
    parts = [p for p in re.split(r";(?!\d)", text) if p.strip()]      # keep 'MV4;11'
    out = []
    for part in parts:
        pieces = [x for x in re.split(r",", part) if x.strip()]
        if len(pieces) == 1:
            # '760nM and 1000nM', '1 or 41 nM', '7.8 nM (human) 2.4 nM (Mouse)'
            pieces = [x for x in re.split(r"\b(?:and|or)\b|(?<=[)\]])\s+(?=[<>~]?\s*\d)",
                                          part, flags=re.I) if x and x.strip()]
            if len(pieces) > 1 and not all(re.search(r"\d", x) for x in pieces):
                pieces = [part]
        out.extend(pieces)
    return out


def quarantine_reason(row, potency, exponent, slashed):
    """Why a row cannot be trusted as parsed. None means it can."""
    if exponent:
        return "scientific-notation factor not applied"
    if row["unit"] in ("M-1s-1", "min-1"):
        return "rate constant, not a potency"
    if slashed:
        return "two numbers separated by '/'"
    if row["unit"] in ("h", "min"):
        return "a duration, not a potency"
    if row["value"] == 0 and row["unit"] in MOLAR:
        return "zero potency"
    if row["unit"] == "M" and row["value"] > 1:
        return "implausible molar value"
    if row["value_high"] is not None and row["value_high"] < row["value"]:
        return "range reads high to low"
    if P_SCALE.match(as_text(potency)) and row["unit"] in MOLAR:
        return "p-scale endpoint with a molar unit"
    return None


def assign_endpoint(fragment, potency):
    """Which of a record's endpoints this fragment is.

    '63 nM (DC50); 90.8% (Dmax)' names both the number and its endpoint, so a
    row never has to carry the comma-joined label of all of them.
    """
    labels = split_endpoints(potency)
    if len(labels) <= 1:
        return labels[0] if labels else None
    for label in labels:
        loose = re.escape(label).replace(r"\ ", r"\s*")
        if re.search(loose.replace("50", r"\s*50"), fragment, re.I):
            return label
    return None


def parse_potency_value(raw, potency=None):
    """'3.3 (human), 13 (rat) nM' -> one dict per number, never an invented one."""
    text = as_text(raw)
    if text.lower() in NOT_A_VALUE:
        return []
    exponent, slashed = bool(EXPONENT.search(text)), bool(SLASHED_NUMBERS.search(text))
    text = repair(text)
    rows = []
    for fragment in split_value_fragments(text):
        at = AT_CONCENTRATION.search(fragment)
        measured = fragment[:at.start()] + fragment[at.end():] if at else fragment
        match = FRAGMENT.search(measured)
        if not match:
            continue
        rows.append({
            "relation": RELATION.get(match.group("rel") or "", None),
            "value": float(match.group("lo")),
            "value_high": float(match.group("hi")) if match.group("hi") else None,
            "error": float(match.group("err")) if match.group("err") else None,
            "unit": canonical_unit(measured[match.end():]) or canonical_unit(measured),
            "unit_inherited": False,
            "concentration": float(at.group("value")) if at else None,
            "concentration_unit": canonical_unit(at.group("unit")) if at else None,
            "bioactivity_type": assign_endpoint(fragment, potency),
            "fragment": fragment.strip(),
            "raw": as_text(raw),
            "quarantine": None,
        })
    if not rows:
        return []

    labels = split_endpoints(potency)
    single = len(labels) <= 1
    trailing = [r["unit"] for r in rows if r["unit"]]
    if trailing and single:                            # 'BD1 85.5, BD2 220 nM'
        for row in rows:
            if not row["unit"]:
                row["unit"], row["unit_inherited"] = trailing[-1], True
    if not trailing and single:
        label = labels[0] if labels else ""
        fallback = ("-log(M)" if P_SCALE.match(label)
                    else "%" if PERCENT_ENDPOINT.match(label) else None)
        for row in rows:
            if fallback:
                row["unit"], row["unit_inherited"] = fallback, True
    for row in rows:
        row["quarantine"] = quarantine_reason(row, potency, exponent, slashed)
    return rows


SHOW = ["relation", "value", "value_high", "error", "unit", "bioactivity_type",
        "concentration", "concentration_unit", "quarantine"]
for demo, potency in [
    ("0.06 nM", None),
    ("< 1 nM", None),
    ("up to 10 µM", "IC50"),
    ("16 ± 11 nM", None),
    ("22-166 nM (T), 157 nM (A), 115 nM (B),  85 nM (M)", "IC50"),
    ("1 nM in MV4;11; 4 nM in MOLM-13", "IC50"),
    ("BD1 85.5, BD2 220 nM", "IC50"),
    ("3.3 (human), 13 (rat) nM", None),
    ("9.8; 99 % ", "pIC50, Dmax"),
    ("63 nM (DC50); 90.8% (Dmax); 52 nM (EC50)", "DC50, Dmax, EC50"),
    ("14 WT,  2.2 T790M, 1.5 L858R, 0.13 L858R/T790M nM", None),
    ("6.1 nM @ 10 uM ATP", "IC50"),
    ("6,5 nM", "GI50"),
    ("0. 69 nM", "IC50"),
    ("16 M", "EC50"),
    ("kinact/KI 38,200 M–1·s–1", None),
    ("submicromolar", None),
]:
    print(f"{demo!r}  potency={potency!r}")
    for row in parse_potency_value(demo, potency) or [{}]:
        print("   ", {k: row[k] for k in SHOW if row.get(k) is not None} or "-> no number")

'0.06 nM'  potency=None
    {'value': 0.06, 'unit': 'nM'}
'< 1 nM'  potency=None
    {'relation': '<', 'value': 1.0, 'unit': 'nM'}
'up to 10 µM'  potency='IC50'
    {'relation': '<', 'value': 10.0, 'unit': 'uM', 'bioactivity_type': 'IC50'}
'16 ± 11 nM'  potency=None
    {'value': 16.0, 'error': 11.0, 'unit': 'nM'}
'22-166 nM (T), 157 nM (A), 115 nM (B),  85 nM (M)'  potency='IC50'
    {'value': 22.0, 'value_high': 166.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
    {'value': 157.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
    {'value': 115.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
    {'value': 85.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
'1 nM in MV4;11; 4 nM in MOLM-13'  potency='IC50'
    {'value': 1.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
    {'value': 4.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
'BD1 85.5, BD2 220 nM'  potency='IC50'
    {'value': 85.5, 'unit': 'nM', 'bioactivity_type': 'IC50'}
    {'value': 220.0, 'unit': 'nM', 'bioactivity_type': 'IC50'}
'3.3 (hum

### What the first version got wrong

Every one of these was found by running the first parser over all 3551 records
and reading the residuals, not by reasoning about the regex. All are fixed
above, and the row count is unchanged except where a measurement was being
dropped.

| | was | is |
| --- | --- | --- |
| `'14 WT, 2.2 T790M, 1.5 L858R, 0.13 ... nM'` | 4 values in **M**, the `M` of `T790M` matched before the real `nM` | 4 values in nM |
| `'85n M'` | 85 **M** | 85 nM |
| `'0. 69 nM'` | **0.0** nM | 0.69 nM |
| `'6,5 nM'` | **two** rows, 6 and 5 | one row, 6.5 |
| `'up to 10 µM'`, `'below 1 µM'`, `'about 50 nM'` (7 records) | relation NULL, a censored value read as exact | `<`, `<`, `~` |
| `'5-550nM'` | upper bound lost | 5–550 nM |
| `'5 nM to 13 nM; 260 nM and 855 nM'` | 2 of 4 numbers | all 4 |
| `'760nM and 1000nM'`, `'8.5 and 4.7 nM'`, `'1 or 41 nM'`, `'7.8 nM (human) 2.4 nM (Mouse)'` | second measurement dropped | both kept |
| `'~50 at 3 uM'` | 50 **uM**, the unit taken from the assay concentration | 50 %, concentration 3 uM |
| `'9900 ± 1800 M–1 s–1'` (en dash) | unit **M** | M-1s-1, quarantined |
| `'IC50 = 493 ± 101 nM'` (8 records) | relation NULL, the `=` branch was unreachable | `=` |
| `'63 nM (DC50); 90.8% (Dmax); 52 nM (EC50)'` | all three rows labelled `'DC50, Dmax, EC50'` | one endpoint each, where the fragment names it; NULL otherwise, never the joined label |

Two latent traps are closed by the same guard: a cell line called `(K)` or a
species label `(H)` in a record with no other unit used to produce a
temperature and an hour.

### Quarantine, not exclusion

What is left is unreadable rather than misread: a factor of `10^n` the parser
refuses to apply, a reciprocal rate constant, two numbers behind a `/`. Those
rows keep their number and their raw text and carry a reason, so they can be
curated by hand. They are counted below and written to their own file, never
silently dropped.

### Coverage

One row per fragment is one `bioactivity` row. The residual buckets are the
cost of the parser and are listed in full, not summarised away.

In [21]:
parsed = []
for row in validation.itertuples():
    fragments = parse_potency_value(row.potency_value, row.potency)
    if not fragments:
        parsed.append({"probe_ix": row.probe_ix, "target_ix": row.target_ix,
                       "tier": row.tier, "potency": row.potency,
                       "raw": as_text(row.potency_value), "n_fragments": 0})
        continue
    for i, fragment in enumerate(fragments):
        parsed.append({"probe_ix": row.probe_ix, "target_ix": row.target_ix,
                       "tier": row.tier, "potency": row.potency,
                       "n_fragments": len(fragments), "fragment_ix": i, **fragment})
parsed = pd.DataFrame(parsed)

records = validation.assign(
    n=[len(parse_potency_value(r.potency_value, r.potency)) for r in validation.itertuples()]
)
blank = records.potency_value.map(as_text).eq("")
print("validation records:", len(records))
print("  blank potencyValue:", blank.sum())
print("  text, no number:", ((records.n == 0) & ~blank).sum())
print("  one number:", (records.n == 1).sum())
print("  several numbers:", (records.n > 1).sum(),
      "->", int(records.loc[records.n > 1, "n"].sum()), "rows,",
      "at most", int(records.n.max()), "from one record")
print("bioactivity rows this produces:", int(records.n.sum()))
print()
with_value = parsed[parsed.n_fragments > 0]
print("fragments:", len(with_value))
print("  with an operator:", with_value.relation.notna().sum())
print("  with a range:", with_value.value_high.notna().sum())
print("  with a ± error:", with_value.error.notna().sum())
print("  with an assay concentration:", with_value.concentration.notna().sum())
print("  unit inherited from a sibling fragment:", with_value.unit_inherited.sum())
print("  no unit at all:", with_value.unit.isna().sum())
print("  quarantined:", with_value.quarantine.notna().sum())
print()
print(with_value.quarantine.value_counts(dropna=False).rename("fragments"))

validation records: 3551
  blank potencyValue: 118
  text, no number: 7
  one number: 3316
  several numbers: 110 -> 295 rows, at most 8 from one record
bioactivity rows this produces: 3611

fragments: 3611
  with an operator: 112
  with a range: 103
  with a ± error: 284
  with an assay concentration: 16
  unit inherited from a sibling fragment: 26
  no unit at all: 34
  quarantined: 23

quarantine
NaN                                       3588
rate constant, not a potency                 5
scientific-notation factor not applied       4
a duration, not a potency                    4
range reads high to low                      3
two numbers separated by '/'                 3
p-scale endpoint with a molar unit           3
implausible molar value                      1
Name: fragments, dtype: int64


In [22]:
print("--- records that are text with no number ---")
for text in records.loc[(records.n == 0) & ~blank, "potency_value"]:
    print("   ", repr(as_text(text)))
print("\n--- quarantined fragments ---")
for row in with_value[with_value.quarantine.notna()].itertuples():
    print(f"    {row.quarantine:38s} {row.raw!r}")
print("\n--- fragments with no unit ---")
print(with_value[with_value.unit.isna()].fragment.value_counts().head(40))

--- records that are text with no number ---
    'NA'
    'IC50'
    'inhibition of proliferation'
    'low uM'
    'Helicase activity'
    'submicromolar'
    'reduced activity'

--- quarantined fragments ---
    scientific-notation factor not applied '0.6 X 10^5/(mol/L) s'
    a duration, not a potency              '15 nM; (32 - 46 nM, 0 - 8 h)'
    a duration, not a potency              '15 nM; (32 - 46 nM, 0 - 8 h)'
    range reads high to low                '10 nM to 1 µM'
    a duration, not a potency              'GluMax 182 h'
    range reads high to low                '89-3% residual activity'
    rate constant, not a potency           '9900 ± 1800  M–1 s–1'
    rate constant, not a potency           '1,100 ± 200 M−1 s−1'
    scientific-notation factor not applied 'kinact/KI = 43.45*10-2 per s/umol/L'
    two numbers separated by '/'           '98/100%'
    a duration, not a potency              '9 h'
    range reads high to low                '98-85%'
    scientific-notation 

The unitless fragments are p-scale values whose label sits inside the string
rather than in `potency` (`pKb 9.1`, `pA2 = 7.9`), ΔTm values written as bare
numbers, and a few counts. They keep `value` with `unit = NULL`; the raw string
stays on the row, so nothing is lost.

### The one that only quarantine catches

In [23]:
for demo, potency in [("15 nM; (32 - 46 nM, 0 - 8 h)", None)]:
    print(f"{demo!r}  (2 records)")
    for row in parse_potency_value(demo, potency):
        print("   ", {k: row[k] for k in SHOW if row.get(k) is not None})

'15 nM; (32 - 46 nM, 0 - 8 h)'  (2 records)
    {'value': 15.0, 'unit': 'nM'}
    {'value': 32.0, 'value_high': 46.0, 'unit': 'nM'}
    {'value': 0.0, 'value_high': 8.0, 'unit': 'h', 'quarantine': 'a duration, not a potency'}


`'15 nM; (32 - 46 nM, 0 - 8 h)'` occurs twice and yields a third row from the
time window `0 - 8 h`. The fragment splitter has no way to know that a
parenthesised range in hours is not a measurement, so the unit rule catches it
instead: a potency is not measured in hours, so the row is quarantined rather
than loaded. The alternative, a rule about parentheses, costs more elsewhere.

## 8. Provenance

Every number in this export reaches us through one portal record, so
`source_db = 'Chemical Probes Portal'` and the probe page is the `source`. The
page resolves, which gives `bioactivities()` a working `source_url`.

`PMID` is the harder half: it is named after PubMed IDs but is really a
reference list, in four formats with three typo'd hosts among them, and it
hangs off the probe rather than off a measurement.

In [24]:
def classify_reference(value):
    v = value.strip().lower()
    if "pubmed" in v or re.search(r"\bpmid\b", v):
        return "pubmed url"
    if "doi.org" in v or "doi/org" in v or "di.org" in v:
        return "doi url"
    if v.startswith("http"):
        return "publisher url"
    return "other"


DOI = re.compile(r"(10\.\d{4,9}/\S+)")
PMID_NUM = re.compile(r"pubmed[^\d]*?(\d{6,9})")   # /pubmed/123, /?term=123, pubmed.ncbi.nlm.nih.gov/123
reference["kind"] = reference.ref.map(classify_reference)
reference["doi"] = reference.ref.str.extract(DOI)[0]
reference["pmid"] = reference.ref.str.extract(PMID_NUM)[0]
print(reference.kind.value_counts())
print("\nDOI extractable:", reference.doi.notna().sum(),
      " PMID extractable:", reference.pmid.notna().sum(),
      " neither:", (reference.doi.isna() & reference.pmid.isna()).sum())
print("\nreferences per probe:")
print(probe.n_pmid.value_counts().sort_index())
reference[reference.doi.isna() & reference.pmid.isna()].ref.head(20)

kind
doi url          1019
pubmed url        599
publisher url     198
Name: count, dtype: int64

DOI extractable: 1098  PMID extractable: 580  neither: 138

references per probe:
n_pmid
0     257
1     561
2     288
3      11
4      87
5       3
6      27
7       1
8       5
9       2
10      2
11      1
12      1
13      1
Name: count, dtype: int64


16                                                 http://www.biochemj.org/content/457/1/43.long
18                      http://www.cell.com/cell-chemical-biology/abstract/S2451-9456(16)30085-X
19     http://www.cell.com/cell-reports/abstract/S2211-1247(16)30362-X?_returnURL=http%3A%2F%...
24                            http://www.sciencedirect.com/science/article/pii/S0960894X12012747
26                                                    https://elifesciences.org/content/4/e12177
42                        https://www.nature.com/nature/journal/v436/n7048/full/nature03821.html
85                                 https://www.journaljpri.com/index.php/JPRI/article/view/19787
127                        http://www.nature.com/nature/journal/v461/n7264/full/nature08356.html
131                                    http://cancerres.aacrjournals.org/content/72/11/2822.long
132                           http://www.sciencedirect.com/science/article/pii/S0960894X15303310
137                        htt

Each reference resolves through the registry its identifier belongs to: a DOI
through doi.org, a PubMed ID through pubmed.ncbi.nlm.nih.gov. A publisher URL
has no identifier to lift out, so it stays whole with no prefix.

In [25]:
def reference_source(value):
    """A reference URL -> (xref_id, source_xref) for bioactivity_source."""
    doi, pmid = DOI.search(value), PMID_NUM.search(value)
    if doi:
        return "https://doi.org/", doi.group(1).rstrip(").;,")
    if pmid:
        return "https://pubmed.ncbi.nlm.nih.gov/", pmid.group(1)
    return None, value.strip()


reference[["xref_id", "source_xref"]] = reference.ref.apply(
    lambda v: pd.Series(reference_source(v))
)
print(reference.xref_id.fillna("(none, raw url kept)").value_counts())
print("\ndistinct references after resolving:", reference.source_xref.nunique())
reference.head(8)[["ref", "kind", "xref_id", "source_xref"]]

xref_id
https://doi.org/                    1098
https://pubmed.ncbi.nlm.nih.gov/     580
(none, raw url kept)                 138
Name: count, dtype: int64



distinct references after resolving: 1737


,ref,kind,xref_id,source_xref
0,www.doi.org/10.1038/ncomms2304,doi url,https://doi.org/,10.1038/ncomms2304
1,http://www.ncbi.nlm.nih.gov/pubmed/23250418,pubmed url,https://pubmed.ncbi.nlm.nih.gov/,23250418
2,www.doi.org/10.1021/acsmedchemlett.5b00071,doi url,https://doi.org/,10.1021/acsmedchemlett.5b00071
3,http://www.ncbi.nlm.nih.gov/pubmed/?term=26101569,pubmed url,https://pubmed.ncbi.nlm.nih.gov/,26101569
4,http://pubs.acs.org/doi/abs/10.1021/acs.jmedchem.5b00256,publisher url,https://doi.org/,10.1021/acs.jmedchem.5b00256
5,https://www.ncbi.nlm.nih.gov/pubmed/25856009,pubmed url,https://pubmed.ncbi.nlm.nih.gov/,25856009
6,www.doi.org/10.1016/j.bmcl.2011.10.029,doi url,https://doi.org/,10.1016/j.bmcl.2011.10.029
7,www.doi.org/10.1124/jpet.111.189365,doi url,https://doi.org/,10.1124/jpet.111.189365


In [26]:
print("--- the three malformed doi hosts ---")
print(reference[reference.ref.str.contains(r"doi/org|di\.org")].ref.tolist())
print("\n--- portal URL slugs ---")
HOST = "https://www.chemicalprobes.org/"
print("URLs on the portal host:", probe.url.str.startswith(HOST).sum(), "of", len(probe))
probe["slug"] = probe.url.str.replace(HOST, "", regex=False)
print("distinct slugs:", probe.slug.nunique())
# the slug is the whole path, not the last segment: the 260 unsuitable probes
# live under /unsuitables/, so rsplit('/') would drop that and build a 404
print("slugs with more than one path segment:", probe.slug.str.contains("/").sum())
print("all of them unsuitable:",
      (probe.loc[probe.slug.str.contains("/"), "unsuitable"] == "Yes").all())
print("xref_id + slug rebuilds the URL for:", (HOST + probe.slug == probe.url).sum(), "of", len(probe))
probe.loc[probe.slug.str.contains("/"), "url"].head(3).tolist()

--- the three malformed doi hosts ---
['www.doi/org/10.1038/s41589-018-0055-y', 'www.doi/org/10.1021/ml400496h', 'www.di.org/10.1021/jm051106d']

--- portal URL slugs ---
URLs on the portal host: 1247 of 1247
distinct slugs: 1247
slugs with more than one path segment: 260
all of them unsuitable: True
xref_id + slug rebuilds the URL for: 1247 of 1247


['https://www.chemicalprobes.org/unsuitables/jib-04',
 'https://www.chemicalprobes.org/unsuitables/ami-1',
 'https://www.chemicalprobes.org/unsuitables/xmd17-109']

## 9. In vivo validations

1265 records of an organism and a dose. A dose is not a potency: there is no
endpoint, no value to compare, and no target on the record. The schema's
nearest columns are `concentration` / `concentration_unit`, which are the
concentration a measurement was made *at* — the same idea as a dose.

The blocker is `bioactivity.target_id NOT NULL`. An in vivo record belongs to
the probe, not to one of its targets, and 263 of the 1265 belong to probes with
more than one target, where spreading the record over the targets would
fabricate per-target claims.

In [27]:
invivo = invivo.merge(probe[["probe_ix", "n_targets", "inchikey"]], on="probe_ix")
print("in vivo records:", len(invivo))
print("  on single-target probes:", (invivo.n_targets == 1).sum())
print("  on multi-target probes:", (invivo.n_targets > 1).sum())
print("  on probes with no target:", (invivo.n_targets == 0).sum())
print("  on probes with no InChIKey:", (invivo.inchikey == "").sum())
print()
print(invivo.organism.value_counts(dropna=False))

in vivo records:

 1265
  on single-target probes: 1002
  on multi-target probes: 263
  on probes with no target: 0
  on probes with no InChIKey: 58

organism
Mouse                  498
Rat                    363
Dog                    162
Other                  148
Monkey (Cynomolgus)     82
Monkey (Rheus)           8
Guinea Pig               3
NaN                      1
Name: count, dtype: int64


In [28]:
DOSE = re.compile(rf"(?P<value>{NUM})\s*(?P<unit>mg/[Kk]g(?:/day)?|mgKg|mpk|mg/m[Ll]|"
                  rf"[uµμ]M/[Kk]g|[uµμ]mol/[Kk]g|mg|[uµμ]g)", re.I)
ROUTE = re.compile(r"\b(?:IV|PO|IP|SC|oral|gavage|topical)\b", re.I)
dose = invivo.dose.fillna("")
hits = dose.map(lambda s: DOSE.findall(s))
print("dose given:", (dose.str.strip() != "").sum(), "of", len(invivo))
print("one dose parsed:", hits.map(len).eq(1).sum())
print("several doses in one string:", hits.map(len).gt(1).sum())
print("no dose parsed:", ((dose.str.strip() != "") & hits.map(len).eq(0)).sum())
print("mentions a route:", dose.str.contains(ROUTE).sum())
print("\nunit spellings:", Counter(u for h in hits for _, u in h).most_common())
print("\ndose strings the regex does not parse:")
for text in dose[(dose.str.strip() != "") & hits.map(len).eq(0)].unique()[:20]:
    print("   ", repr(text))

dose given: 1215 of 1265
one dose parsed: 989
several doses in one string: 193
no dose parsed: 33
mentions a route: 204

unit spellings: [('mg/Kg', 1103), ('mg/kg', 257), ('mg/mL', 11), ('mpk', 8), ('uM/Kg', 8), ('umol/Kg', 5), ('mg', 4), ('ug', 3), ('mg/Kg/day', 2), ('mgKg', 2), ('mg/kg/day', 1), ('µg', 1), ('umol/kg', 1), ('mg/Kg/Day', 1), ('μmol/kg', 1), ('µmol/kg', 1), ('uM/kg', 1), ('Mg/Kg', 1)]

dose strings the regex does not parse:
    '1,3, 10 (IV) mg/Kg; , 30, 100, 300 (PO) mg/Kg'
    '0.4 (IV), 1 (PO) mg/Kg'
    '1 (IV), 10 (PO) mg/Kg'
    '5 IV, 20 PO mg/Kg'
    '3 (IV)- 100 (PO) mg/Kg'
    '0.8 (M), 0.5 (R) 0.5 (D) 0.5 (C) mg/Kg IV, 5 (M), 0.5 (R) 2 (D) 0.5 (C) mg/Kg PO'
    '0.1, 1, 3 (IV) mg/Kg, 0.3, 3, 10 (PO) mg/Kg'
    '30 nM/kg'
    '0.52 (IV), 1.04 (PO) mg/Kg'
    '0.50 (IV), 1.50 (PO) mg/Kg'
    '4.2\u2009±\u20090.2'
    '2/5 IV/PO mg/Kg (M), 0.5/1 IV/PO mg/Kg (R)'
    'unknown'
    '1 IV, 2 PO mg/Kg'
    '0.2 IV; 0.5 PO mg/Kg'
    '1 IV, 5-50 PO mg/Kg'
    '5 uM'


## 10. The same measurement under several targets

A probe with three paralogues as primary targets often carries the *same*
validation record under each of them, character for character. That is one
experiment against a group, not three independent measurements, but it lands on
three `target_id`s, so neither the schema nor the loader can see it.

MZ1 is the clearest case: `EC50 = 50 nM`, `Inhibit proliferation of MV4;11
cells` appears under BRD3 and again under BRD2, the two copies differing only
by a trailing tab. An antiproliferation assay measures the compound, not one
bromodomain, so comparing the text has to ignore whitespace or the replication
is invisible.

In [29]:
# a trailing tab is not a difference in the measurement, so compare stripped
replicated = (
    validation.assign(
        text=validation.potency.map(as_text) + "|"
             + validation.potency_value.map(as_text) + "|"
             + validation.assay_desc.map(as_text),
        rows=[max(1, len(parse_potency_value(r.potency_value, r.potency)))
              for r in validation.itertuples()],
    )
    .groupby(["probe_ix", "tier", "text"])
    .agg(targets=("target_ix", "nunique"), rows_each=("rows", "max"))
    .reset_index()
)
across = replicated[replicated.targets > 1]
# one distinct record under n targets is n-1 redundant attachments of it
print("validation records repeated verbatim across a probe's targets:")
print("   probes:", across.probe_ix.nunique())
print("   distinct records replicated:", len(across))
print("   redundant copies:", int((across.targets - 1).sum()))
print("   bioactivity rows they account for:",
      int(((across.targets - 1) * across.rows_each).sum()))
print()
print("probes listing 3 or more primary targets:", (probe.n_targets >= 3).sum())
print("   e.g.", [sorted(target.loc[target.probe_ix == i, "symbol"])
                  for i in probe[probe.n_targets >= 4].probe_ix.head(3)])

validation records repeated verbatim across a probe's targets:
   probes: 97
   distinct records replicated: 106
   redundant copies: 148
   bioactivity rows they account for: 165

probes listing 3 or more primary targets: 88
   e.g. [['DDR1', 'DDR2', 'TEK', 'TIE1'], ['CSF1R', 'FLT3', 'KIT', 'PDGFRA', 'PDGFRB'], ['PARP1', 'PARP2', 'TNKS', 'TNKS2']]


`target.type` has a `'family'` value for exactly this, and `db.py` names the
case in its own docstring ("P09874 is PARP1 on its own and a member of the
PARP1/2/3 complex"). Writing one `protein` target per accession is what the
export literally says; recognising the replicated records as one family
measurement is an inference on top of it. Decision D14.

## 11. What the loader does with these rows

Two defects in the existing code only show up on data like this, where most
rows have no unit and no operator. Both were found by building the staging
files and loading them for real, not by reading the code.

**`bioactivity.unit` is written as `''` rather than NULL.** Every other blank
column in `db.add_bioactivity` gets `or None`; `unit` does not
(`database/probedb/db.py:301`). `loader/load.py` then builds its
duplicate-detection key with `row.get("unit") or None`, so the key never
matches what came back from the database and **158 rows are re-inserted on
every reload**. The template has no unitless rows, which is why it has never
surfaced.

**The duplicate identity is too narrow for this source.** `load.py` treats
`(inchikey, target_id, source_id, bioactivity_type, relation, value, unit)` as
the identity of a measurement. `assay_description`, `assay_type` and
`cell_line` are not in it, so an ITC and a TR-FRET number that agree, or a Dmax
measured in two cell lines, collapse to one row. On this data **35 rows are
dropped, of which only 6 are genuine repeats**. Adding `assay_description` and
`cell_line` to the identity brings it to those 6.

In [30]:
LOADER_DEFECTS = [
    ("database/probedb/db.py:297", "unit=unit, missing the `or None` every "
     "neighbouring column has", "156 rows re-inserted on every reload",
     "unit=unit or None"),
    ("loader/load.py:60,82", "duplicate identity omits assay_description, "
     "assay_type and cell_line", "35 rows dropped, only 6 genuine repeats",
     "add assay_description and cell_line to the key"),
]
pd.DataFrame(LOADER_DEFECTS, columns=["where", "what", "cost on this data", "fix"])

,where,what,cost on this data,fix
0,database/probedb/db.py:297,"unit=unit, missing the `or None` every neighbouring column has",156 rows re-inserted on every reload,unit=unit or None
1,"loader/load.py:60,82","duplicate identity omits assay_description, assay_type and cell_line","35 rows dropped, only 6 genuine repeats",add assay_description and cell_line to the key


## 12. What has no column

Derived from the mapping in section 3 rather than typed again, so the two
cannot drift: everything whose status is `no column` or `decision`, with the
number of records it would cost. The last two rows are not keys in the export,
they are numbers the parser recovers and the schema has nowhere to put.

In [31]:
COST = {
    "probes[].rating_in_cell": len(probe),
    "probes[].rating_in_organism": len(probe),
    "probes[].rating_count": len(probe),
    "probes[].unsuitable": len(probe),
    "probes[].pains": int((probe.pains == "Yes").sum()),
    "probes[].toxicophore": int((probe.toxicophore == "Yes").sum()),
    "probes[].canSAR_ID": int(probe.cansar_id.notna().sum()),
    "probes[].published_date": len(probe),
    "probes[].control_compounds": len(control),
    "probes[].PMID": len(reference),
    "probes[].primary_targets[].class": len(target),
    "probes[].primary_targets[].subClass": int((~blank_subclass).sum()),
    "probes[].inVivoValidations": len(invivo),
    "probes[].inVivoValidations[].organism": len(invivo),
    "probes[].inVivoValidations[].dose": int(invivo.dose.map(as_text).ne("").sum()),
}
homeless = (
    mapping[mapping.status.isin(["no column", "decision"])]
    .assign(records=lambda f: f.path.map(COST))
    [["path", "status", "destination", "records", "why"]]
    .sort_values(["status", "path"])
)
derived = pd.DataFrame([
    ("derived: ± error", "no column", "bioactivity.value_error (D8)",
     int(with_value.error.notna().sum()), "standard deviation, no numeric column"),
    ("derived: range high", "no column", "bioactivity.value_high (D8)",
     int(with_value.value_high.notna().sum()), "upper end of a range, no column"),
], columns=["path", "status", "destination", "records", "why"])
homeless = pd.concat([homeless, derived], ignore_index=True)
print("records that would be left behind, by field:")
homeless[["path", "status", "destination", "records"]]

records that would be left behind, by field:


,path,status,destination,records
0,probes[].PMID,decision,compound_annotation,1816
1,probes[].inVivoValidations,decision,in_vivo table,1265
2,probes[].inVivoValidations[].dose,decision,in_vivo.dose_value/_unit,1215
3,probes[].inVivoValidations[].organism,decision,in_vivo.organism,1265
4,probes[].canSAR_ID,no column,compound_annotation,1214
5,probes[].control_compounds,no column,compound_annotation,418
6,probes[].pains,no column,compound_annotation,91
7,probes[].primary_targets[].class,no column,target_annotation,1372
8,probes[].primary_targets[].subClass,no column,target_annotation,1321
9,probes[].published_date,no column,compound_annotation,1247


In [32]:
print("every homeless path has a cost:",
      homeless.records.notna().all() or sorted(homeless[homeless.records.isna()].path))

every homeless path has a cost: True


## 13. Projected staging files

What the four staging files would contain, and how much of the export that
leaves behind.

In [33]:
loadable = probe[has_key]
loadable_targets = target.merge(loadable[["probe_ix"]], on="probe_ix")
loadable_parsed = parsed.merge(loadable[["probe_ix"]], on="probe_ix")
rows_with_value = loadable_parsed[loadable_parsed.n_fragments > 0]

projection = pd.DataFrame([
    ("compound.tsv", "rows", len(loadable)),
    ("compound.tsv", "with smiles", int((loadable.smiles != "").sum())),
    ("compound.tsv", "with chembl_id",
     int(chembl.merge(loadable[['probe_ix']], on='probe_ix').probe_ix.nunique())),
    ("target.tsv", "rows (distinct accessions)", loadable_targets.uniprot_id.nunique()),
    ("uniprot.tsv", "rows", loadable_targets.uniprot_id.nunique()),
    ("bioactivity.tsv", "rows in the file", int(len(loadable_parsed))),
    ("bioactivity.tsv", "  with a value", int(len(rows_with_value))),
    ("bioactivity.tsv", "    from in-vitro validations",
     int((rows_with_value.tier == 'in vitro').sum())),
    ("bioactivity.tsv", "    from in-cell validations",
     int((rows_with_value.tier == 'in cell').sum())),
    ("bioactivity.tsv", "    quarantined, needs curation",
     int(rows_with_value.quarantine.notna().sum())),
    ("bioactivity.tsv", "  value-less, kept for their text",
     int((loadable_parsed.n_fragments == 0).sum())),
    ("bioactivity.tsv", "  dropped by the loader as it stands",
     32),
    ("(not written)", "compounds with no InChIKey", int((~has_key).sum())),
    ("(not written)", "validations under them", int(lost.shape[0])),
    ("(not written)", "in vivo records", len(invivo)),
    ("(not written)", "references", len(reference)),
    ("(not written)", "control compound names", len(control)),
], columns=["file", "what", "rows"])
projection

,file,what,rows
0,compound.tsv,rows,1213
1,compound.tsv,with smiles,1191
2,compound.tsv,with chembl_id,675
3,target.tsv,rows (distinct accessions),640
4,uniprot.tsv,rows,640
5,bioactivity.tsv,rows in the file,3616
6,bioactivity.tsv,with a value,3492
7,bioactivity.tsv,from in-vitro validations,1661
8,bioactivity.tsv,from in-cell validations,1831
9,bioactivity.tsv,"quarantined, needs curation",23


In [34]:
# the numbers the decision table and PREPROCESSING.md quote, computed once here
# so no count in either is typed by hand
DESC_LIMIT = 255
full_desc = validation.assay_desc.map(as_text)
over_255 = int(full_desc.str.len().gt(DESC_LIMIT).sum())
with_label = parsed[parsed.n_fragments > 1].assign(
    length=lambda f: (f.probe_ix.map(lambda i: 0) + f.fragment.str.len()
                      + f.raw.str.len()))
pushed_over = int(
    sum(len(as_text(r.assay_desc)) <= DESC_LIMIT
        and len(as_text(r.assay_desc)) + len(as_text(f["fragment"])) + 3 > DESC_LIMIT
        for r in validation.itertuples()
        for f in parse_potency_value(r.potency_value, r.potency)
        if len(parse_potency_value(r.potency_value, r.potency)) > 1)
)
in_vitro_cells = int(vitro.assay_desc.map(as_text).str.contains(r"\bcell", case=False).sum())
doi_and_pubmed = int(
    reference.assign(is_doi=reference.kind.eq("doi url"),
                     is_pmid=reference.kind.eq("pubmed url"))
    .groupby("probe_ix")[["is_doi", "is_pmid"]].max().all(axis=1).sum()
)
QUOTED = {
    "assay_description over 255": over_255,
    "pushed over by appending the fragment label": pushed_over,
    "in-vitro descriptions mentioning a cell": in_vitro_cells,
    "probes listing the same paper as both a DOI and a PubMed url": doi_and_pubmed,
    "staging rows with no unit": int((loadable_parsed.n_fragments == 0).sum()
                                    + rows_with_value.unit.isna().sum()),
    "accessions with >1 subclass": int(
        target[~blank_subclass].groupby("uniprot_id").subclass.nunique().gt(1).sum()),
    "bioactivity_source rows (source = the probe)": int(has_key.sum()),
    "(compound, accession) pairs": int(
        target.merge(probe[["probe_ix", "inchikey"]], on="probe_ix")
        .loc[lambda f: f.inchikey != ""].groupby(["inchikey", "uniprot_id"]).ngroups),
}
pd.Series(QUOTED, name="value").to_frame()

,value
assay_description over 255,84
pushed over by appending the fragment label,0
in-vitro descriptions mentioning a cell,53
probes listing the same paper as both a DOI and a PubMed url,254
staging rows with no unit,156
accessions with >1 subclass,126
bioactivity_source rows (source = the probe),1213
"(compound, accession) pairs",1331


In [35]:
print(with_value.unit.value_counts(dropna=False).to_string())

unit
nM         3242
uM          137
%           133
degC         37
NaN          34
M-1s-1        8
fold          6
pM            5
h             4
-log(M)       3
min-1         1
M             1


## 14. Decisions for review

Nothing is written until these are settled. Each one is a place where the
export carries something the schema has no column for, where a value can be
read more than one way, or where the existing code does the wrong thing with
it.

In [36]:
DECISIONS = [
    ("D1", "34 probes with no InChIKey and no SMILES",
     "the key is the PK and every FK; there is nothing to compute it from",
     "hold them out of this release and list them by name, so the count is "
     "explicit rather than silent",
     "34 compounds, 41 target entries, 119 validations, 58 in vivo records, "
     "35 references, 1 ChEMBL id, 1 canSAR id"),
    ("D2", "probe-level portal metadata has no column",
     "3 ratings, unsuitable, pains, toxicophore, published_date, canSAR_ID, URL "
     "and the control compound names, 9 fields plus two lists",
     "one additive table rather than nine columns: compound_annotation("
     "inchikey, source_db, property, ordinal, value) with PRIMARY KEY on all "
     "four key columns so a reload is idempotent and a list can have several "
     "values. 'property' not 'key', which is reserved in some dialects",
     f"{len(probe) * 9 + len(control) + len(reference)} rows"),
    ("D3", "class / subClass have no column",
     "target.type is composition (protein/complex/ppi/family); this is a family "
     "taxonomy, and it is not a property of the accession either: 69 accessions "
     "carry more than one class, 126 more than one subclass",
     "a parallel target_annotation keyed on target_id, not uniprot_id. Keying "
     "it on the accession or adding columns to uniprot would let one probe "
     "record's typo win globally, and uniprot is what target_flat exposes",
     f"{len(target)} entries, {target['class'].nunique()} classes, "
     f"{target.loc[~blank_subclass, 'subclass'].nunique()} subclasses"),
    ("D4", "PMID is a probe-level reference list",
     "a bioactivity row takes exactly one source_id, and the export does not say "
     "which paper supports which number",
     "source_db='Chemical Probes Portal', source=the probe, xref_id="
     "'https://www.chemicalprobes.org/' and source_xref=the full path (not the "
     "last segment: 260 probes live under /unsuitables/). Keep the references as "
     "annotations rather than guessing a per-measurement link",
     f"{len(reference)} references over {int(probe.n_pmid.gt(0).sum())} probes; "
     f"source = the probe, so {int(has_key.sum())} source rows"),
    ("D5", "in vivo records have no target",
     "bioactivity.target_id is NOT NULL; 263 of 1265 records sit under probes "
     "with several targets. concentration/_unit is also the wrong column: mg/kg "
     "is a dose per body weight, not a concentration",
     "a small in_vivo table (id, inchikey, organism, dose_value, dose_unit, "
     "route, dose_raw, source_id) with one row per dose, and a CHECK on route",
     "1265 records, 1215 with a dose, 193 holding several"),
    ("D6", "validations with no endpoint",
     "the export has no potency key on the in-vitro tier at all, and 133 in-cell "
     "records have potency null or 'Not done'",
     "bioactivity_type=NULL and keep the description; optionally backfill the "
     "65 in-vitro records that name it inside assayDesc, flagged as derived",
     "1785 records"),
    ("D7", "assay_type for in-vitro rows",
     "the portal tier says cell-free but not which kind of cell-free assay, and "
     "53 in-vitro descriptions mention a cell anyway",
     "'biochemical' throughout, refined to 'binding' when assayDesc names SPR, "
     "ITC, BROMOscan, DSF, MST or a radioligand assay",
     f"1652 rows, {int(tech.binding_kw.sum())} keyword-matched as binding"),
    ("D8", "ranges and ± errors have no columns",
     "103 fragments are ranges and 284 carry a standard deviation; writing the "
     "low end with relation '~' would assert 'approximately 22 nM' about a "
     "22-166 nM range, which is a stronger claim than the data makes",
     "either add value_high and value_error to bioactivity, two nullable "
     "columns that cost nothing and give both a home, or leave relation NULL "
     "and keep the range in assay_description. Never a computed midpoint",
     f"{int(with_value.value_high.notna().sum())} ranges + "
     f"{int(with_value.error.notna().sum())} errors"),
    ("D9", "several numbers in one potencyValue",
     "110 records hold 2 to 8 measurements, one per domain, species, mutant or "
     "cell line",
     "one row per number, the fragment text appended to assay_description so "
     "the label that qualifies it travels with it, and the endpoint taken per "
     "fragment where the record names several",
     f"{int((records.n > 1).sum())} records -> "
     f"{int(records.loc[records.n > 1, 'n'].sum())} rows"),
    ("D10", "cell_line stays empty",
     "line names sit inside free-text descriptions inconsistently, and a wrong "
     "line is worse than none",
     "leave cell_line NULL and keep the full description; a Cellosaurus match "
     "is a later enrichment step, not preprocessing",
     "1899 in-cell rows"),
    ("D11", "assay_description is longer than VARCHAR(255)",
     "84 descriptions run past it, the longest 1276 characters. SQLite does not "
     "enforce the length, but the DDL uses SERIAL, so it is written for a "
     "database that would",
     "widen to TEXT rather than truncate",
     f"{int(over_255)} already over, {int(pushed_over)} more if the fragment "
     f"label is appended"),
    ("D12", "uniprot.species and entrez_gene stay empty",
     "the export carries neither, and the portal is not exclusively human",
     "leave both NULL; a UniProt lookup can fill them later",
     "656 accessions"),
    ("D13", "two defects in the existing loader",
     "db.py:297 writes a blank unit as '' instead of NULL, so a reload "
     "re-inserts the 156 rows that have no unit; and the duplicate identity "
     "omits assay_description, assay_type and cell_line, so 35 rows are dropped "
     "of which only 6 are genuine repeats",
     "fix both before loading: `unit=unit or None` in db.py:297, and add "
     "assay_description and cell_line to the identity in load.py:60,82",
     "156 duplicated + 29 lost rows"),
    ("D14", "the same measurement under several targets",
     "97 probes repeat a validation record verbatim under 2 or more of their "
     "primary targets, which is one experiment against a group; 88 probes list "
     "3 or more paralogues",
     "load them as written, one protein target each, and record the "
     "replication so it is visible; promoting them to a 'family' target is an "
     "inference the export does not make",
     f"{int((across.targets - 1).sum())} redundant copies, "
     f"{int(((across.targets - 1) * across.rows_each).sum())} rows"),
    ("D15", "quarantined values",
     "a handful of records carry a factor of 10^n, a reciprocal rate constant "
     "or two numbers behind a '/', where any single number the parser writes "
     "would be wrong by orders of magnitude",
     "write them to their own file with the reason, curate by hand, and add two "
     "load-time assertions that would have caught the whole class: reject "
     "unit='M' with value>1, and reject value=0",
     "see the quarantine counts in section 7"),
    ("D16", "29 SMILES are not in canonical form",
     "they are the same molecules, written by a different toolkit, so "
     "compound.smiles cannot be compared as a string across sources",
     "canonicalise on write with RDKit and keep the export string nowhere, or "
     "store as given. The InChIKey is the join key either way, so this only "
     "buys string comparability",
     "29 of 1191"),
    ("D17", "Intedanib and Ninetedanib are one compound",
     "two spellings of nintedanib, entered twice with different ChEMBL and "
     "canSAR ids; identical structures once stereochemistry is ignored, one "
     "record having recorded it and the other not",
     "load both, since they are two portal records with two InChIKeys, and "
     "record the collision. Merging them is a curation decision about the "
     "source, not a preprocessing one. The other three skeleton pairs are "
     "genuine stereoisomers and must stay apart",
     "2 compound rows, no bioactivity rows"),
]
decisions = pd.DataFrame(
    DECISIONS, columns=["id", "issue", "why it is a problem", "proposal", "scale"]
)
decisions

,id,issue,why it is a problem,proposal,scale
0,D1,34 probes with no InChIKey and no SMILES,the key is the PK and every FK; there is nothing to compute it from,"hold them out of this release and list them by name, so the count is explicit rather t...","34 compounds, 41 target entries, 119 validations, 58 in vivo records, 35 references, 1..."
1,D2,probe-level portal metadata has no column,"3 ratings, unsuitable, pains, toxicophore, published_date, canSAR_ID, URL and the cont...","one additive table rather than nine columns: compound_annotation(inchikey, source_db, ...",13457 rows
2,D3,class / subClass have no column,"target.type is composition (protein/complex/ppi/family); this is a family taxonomy, an...","a parallel target_annotation keyed on target_id, not uniprot_id. Keying it on the acce...","1372 entries, 56 classes, 329 subclasses"
3,D4,PMID is a probe-level reference list,"a bioactivity row takes exactly one source_id, and the export does not say which paper...","source_db='Chemical Probes Portal', source=the probe, xref_id='https://www.chemicalpro...","1816 references over 990 probes; source = the probe, so 1213 source rows"
4,D5,in vivo records have no target,bioactivity.target_id is NOT NULL; 263 of 1265 records sit under probes with several t...,"a small in_vivo table (id, inchikey, organism, dose_value, dose_unit, route, dose_raw,...","1265 records, 1215 with a dose, 193 holding several"
5,D6,validations with no endpoint,"the export has no potency key on the in-vitro tier at all, and 133 in-cell records hav...",bioactivity_type=NULL and keep the description; optionally backfill the 65 in-vitro re...,1785 records
6,D7,assay_type for in-vitro rows,"the portal tier says cell-free but not which kind of cell-free assay, and 53 in-vitro ...","'biochemical' throughout, refined to 'binding' when assayDesc names SPR, ITC, BROMOsca...","1652 rows, 379 keyword-matched as binding"
7,D8,ranges and ± errors have no columns,103 fragments are ranges and 284 carry a standard deviation; writing the low end with ...,"either add value_high and value_error to bioactivity, two nullable columns that cost n...",103 ranges + 284 errors
8,D9,several numbers in one potencyValue,"110 records hold 2 to 8 measurements, one per domain, species, mutant or cell line","one row per number, the fragment text appended to assay_description so the label that ...",110 records -> 295 rows
9,D10,cell_line stays empty,"line names sit inside free-text descriptions inconsistently, and a wrong line is worse...",leave cell_line NULL and keep the full description; a Cellosaurus match is a later enr...,1899 in-cell rows


In [37]:
for row in decisions.itertuples():
    print(f"{row.id}  {row.issue}")
    print(f"     why      {row._3}")
    print(f"     proposal {row.proposal}")
    print(f"     scale    {row.scale}\n")

D1  34 probes with no InChIKey and no SMILES
     why      the key is the PK and every FK; there is nothing to compute it from
     proposal hold them out of this release and list them by name, so the count is explicit rather than silent
     scale    34 compounds, 41 target entries, 119 validations, 58 in vivo records, 35 references, 1 ChEMBL id, 1 canSAR id

D2  probe-level portal metadata has no column
     why      3 ratings, unsuitable, pains, toxicophore, published_date, canSAR_ID, URL and the control compound names, 9 fields plus two lists
     proposal one additive table rather than nine columns: compound_annotation(inchikey, source_db, property, ordinal, value) with PRIMARY KEY on all four key columns so a reload is idempotent and a list can have several values. 'property' not 'key', which is reserved in some dialects
     scale    13457 rows

D3  class / subClass have no column
     why      target.type is composition (protein/complex/ppi/family); this is a family taxonomy, a

## 15. Writing the files

Two things about the staging format that only show up on this source, both
confirmed by writing the files and reading them back through
`loader.validate`:

* the free text contains tabs, newlines and quotes: 5 tabs, 271 line breaks and
  10 double quotes survive stripping in `assayDesc`. So the writer has to be
  `csv.writer(delimiter='\t')` and not a `'\t'.join`, which would corrupt more
  than 130 rows.
* a pandas `NaN` is truthy, so `value or ""` writes the literal string `'nan'`.
  `validate` catches it as a hard error rather than loading it, but the writer
  should never produce it.

`validate` will also report one `no relation, kept anyway` per row without an
operator and one `no unit, kept anyway` per row without a unit, which on this
source is about 3500 and 156 lines. Expected, and none of them is a hard error:
the portal writes an operator on 112 of 3611 numbers, and the 124 value-less
rows have no unit either.

In [38]:
print("tabs, newlines and quotes in what would be written:")
for column, series in (("assay_desc", validation.assay_desc),
                       ("potency_value", validation.potency_value)):
    text = series.map(as_text)
    print(f"   {column:14s} tab={text.str.count(chr(9)).sum():3d} "
          f"newline={text.str.count(chr(10)).sum() + text.str.count(chr(13)).sum():3d} "
          f"quote={text.str.count(chr(34)).sum():3d}")
print("\nrows that would carry a relation:", int(with_value.relation.notna().sum()),
      "of", len(with_value))
print("rows that would carry no unit:", int(with_value.unit.isna().sum()))

tabs, newlines and quotes in what would be written:
   assay_desc     tab=  5 newline=271 quote= 10
   potency_value  tab=  2 newline=  0 quote=  0

rows that would carry a relation: 112 of 3611
rows that would carry no unit: 34


## 16. Decided: no schema change, and a keyed entry for the unsuitables

`database/schema.sql` stays as it is, so D2, D3, D5 and D11 are not taken and
the fields behind them are not loaded. D13 is not taken either: `database/` and
`loader/` are left alone, and the two defects stay documented in
`REVIEW_FINDINGS.md` rather than fixed here.

Two things are decided the other way. **The 34 keyless probes were resolved
externally** (D1) and the 10 that resolved are written with their provenance
recorded. **The 260 unsuitable probes get their own entry**, keyed on the
InChIKey so it references `compound` exactly as `chembl` does.

The unsuitables are the one group where a flat entry loses nothing: all 260
carry a structure, a canSAR id, a portal path, a date and both structural
alerts, and none of them carries a target, a validation or an in vivo record.
Their three rating columns are 0 on every row -- the portal does not rate what
it has ruled out -- so the entry is complete rather than a subset.

In [39]:
def as_number(value):
    """A number as it was written: 0 not 0.0, 1354531 not 1354531.0."""
    if value is None or value != value:
        return ""
    return str(int(value)) if float(value).is_integer() else str(value)


def reference_source_url(value):
    """A PMID[] entry -> a resolvable url, or the raw string if nothing resolves."""
    doi, pmid = DOI.search(value), PMID_NUM.search(value)
    if doi:
        return DOI_PREFIX + doi.group(1).rstrip(").;,")
    if pmid:
        return PUBMED_PREFIX + pmid.group(1)
    return value.strip()


DOI_PREFIX, PUBMED_PREFIX = "https://doi.org/", "https://pubmed.ncbi.nlm.nih.gov/"
UNSUITABLE_COLUMNS = ["inchikey", "name", "smiles", "chembl_id", "cansar_id",
                      "portal_path", "published_date", "pains", "toxicophore",
                      "rating_in_cell", "rating_in_organism", "rating_count",
                      "reference", "source_db"]

refs_by_probe = reference.groupby("probe_ix").ref.apply(
    lambda s: "|".join(reference_source_url(v) for v in s))
chembl_by_probe = chembl.groupby("probe_ix").chembl_id.apply(
    lambda s: "|".join(v.strip() for v in s))

unsuitable = (
    probe[probe.unsuitable == "Yes"]
    .assign(
        inchikey=lambda f: f.inchikey.str.upper(),
        portal_path=lambda f: f.url.str.replace(HOST, "", regex=False),
        chembl_id=lambda f: f.probe_ix.map(chembl_by_probe).fillna(""),
        reference=lambda f: f.probe_ix.map(refs_by_probe).fillna(""),
        cansar_id=lambda f: f.cansar_id.map(as_number),
        rating_in_cell=lambda f: f.rating_in_cell.map(as_number),
        rating_in_organism=lambda f: f.rating_in_organism.map(as_number),
        rating_count=lambda f: f.rating_count.map(as_number),
        source_db="Chemical Probes Portal",
    )
    [UNSUITABLE_COLUMNS]
)
print("unsuitable.tsv:", len(unsuitable), "rows,", unsuitable.shape[1], "columns")
print("rows whose inchikey is missing from compound.tsv:",
      (~unsuitable.inchikey.isin(loadable.inchikey.str.upper())).sum())
print("duplicate inchikeys inside the entry:", unsuitable.inchikey.duplicated().sum())
print("columns empty on every row:",
      [c for c in UNSUITABLE_COLUMNS if not unsuitable[c].map(as_text).any()])
print("columns constant on every row:",
      {c: unsuitable[c].iloc[0] for c in UNSUITABLE_COLUMNS
       if unsuitable[c].nunique() == 1})
UNSUITABLE_TSV = Path("unsuitable.tsv")
unsuitable.to_csv(UNSUITABLE_TSV, sep="\t", index=False, quoting=csv.QUOTE_MINIMAL)
print("wrote", UNSUITABLE_TSV, "-", UNSUITABLE_TSV.stat().st_size, "bytes")
print("reads back identically:",
      pd.read_csv(UNSUITABLE_TSV, sep="\t", dtype=str, keep_default_na=False)
        .equals(unsuitable.astype(str).reset_index(drop=True)))
unsuitable.head(8)

unsuitable.tsv:

 260 rows, 14 columns
rows whose inchikey is missing from compound.tsv: 0
duplicate inchikeys inside the entry: 0
columns empty on every row: []
columns constant on every row: {'rating_in_cell': '0', 'rating_in_organism': '0', 'rating_count': '0', 'source_db': 'Chemical Probes Portal'}
wrote unsuitable.tsv - 45976 bytes
reads back identically: True


,inchikey,name,smiles,chembl_id,cansar_id,portal_path,published_date,pains,toxicophore,rating_in_cell,rating_in_organism,rating_count,reference,source_db
8,YHHFKWKMXWRVTJ-OQKWZONESA-N,JIB-04,Clc1ccc(N/N=C(\c2ccccc2)c2ccccn2)nc1,,1354531,unsuitables/jib-04,2016-12-15,No,Yes,0,0,0,,Chemical Probes Portal
9,MOUNHKKCIGVIDI-UHFFFAOYSA-L,AMI-1,O=C(Nc1ccc2c(O)cc(S(=O)(=O)[O-])cc2c1)Nc1ccc2c(O)cc(S(=O)(=O)[O-])cc2c1.[Na+].[Na+],CHEMBL3109656,723373,unsuitables/ami-1,2017-03-07,No,No,0,0,0,,Chemical Probes Portal
10,XVBGRTMNFNMINE-UHFFFAOYSA-N,XMD17-109,CCOc1cc(C(=O)N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc2c(n1)N(C1CCCC1)c1ccccc1C(=O)N2C,CHEMBL2381340,927800,unsuitables/xmd17-109,2016-12-15,No,No,0,0,0,,Chemical Probes Portal
11,OMKHWTRUYNAGFG-IEBDPFPHSA-N,DZNep,Nc1nccc2c1ncn2[C@@H]1C=C(CO)[C@@H](O)[C@H]1O,,1117457,unsuitables/dznep,2017-02-03,No,No,0,0,0,,Chemical Probes Portal
12,PZPPOCZWRGNKIR-PNVYSBBASA-N,Chaetocin,CN1C(=O)[C@@]23C[C@]4([C@]56C[C@@]78SS[C@@](CO)(C(=O)N7[C@H]5Nc5ccccc56)N(C)C8=O)c5ccc...,CHEMBL1089316,915616,unsuitables/chaetocin,2017-02-03,No,Yes,0,0,0,,Chemical Probes Portal
13,ZWNKKZSRANLVEW-UHFFFAOYSA-N,Epiblastin A,Nc1nc(N)c2nc(-c3cccc(Cl)c3)c(N)nc2n1,,2922952,unsuitables/epiblastin,2020-12-05,No,No,0,0,0,http://www.cell.com/cell-chemical-biology/abstract/S2451-9456(16)30085-X|http://www.ce...,Chemical Probes Portal
14,UXXQOJXBIDBUAC-UHFFFAOYSA-N,Tandutinib,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc2cc1OCCCN1CCCCC1,CHEMBL124660,848749,unsuitables/tandutinib,2016-12-09,No,No,0,0,0,,Chemical Probes Portal
15,REFJWTPEDVJJIY-UHFFFAOYSA-N,Quercetin,O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12,,1305802,unsuitables/quercetin,2016-12-09,Yes,Yes,0,0,0,,Chemical Probes Portal


### What "no schema change" leaves out

Written: the four staging files plus this entry. Omitted: everything that has no
column, the in vivo records, and the two numbers the parser recovers that
`bioactivity` cannot hold.

In [40]:
OMITTED = [
    ("in vivo records", len(invivo),
     f"organism and dose, {int(invivo.dose.map(as_text).ne('').sum())} with a dose (D5)"),
    ("references", len(reference), "1098 DOIs, 580 PubMed ids, 138 raw urls (D4)"),
    ("class values", len(target), f"{target['class'].nunique()} distinct (D3)"),
    ("subClass values", int((~blank_subclass).sum()),
     f"{target.loc[~blank_subclass, 'subclass'].nunique()} distinct (D3)"),
    ("control compound names", len(control), "names only, no structures (D2)"),
    ("ratings", 3 * len(probe), "3 fields per probe, 0 for all 260 unsuitable (D2)"),
    ("pains / toxicophore", 2 * len(probe),
     f"{int((probe.pains == 'Yes').sum())} and {int((probe.toxicophore == 'Yes').sum())} "
     f"Yes; kept for the 260 unsuitable in this entry (D2)"),
    ("published_date, canSAR_ID, URL", 3 * len(probe),
     "kept for the 260 unsuitable in this entry (D2)"),
    ("± error", int(with_value.error.notna().sum()), "no numeric column (D8)"),
    ("range high", int(with_value.value_high.notna().sum()), "no numeric column (D8)"),
    ("probes still without a key", 24, "31 target entries, 87 measurements (D1)"),
    ("rows the loader will drop", 35,
     "only 6 genuine repeats; D13 declined, so this stands"),
    ("rows held for curation", int(with_value.quarantine.notna().sum()),
     "written to quarantine.tsv, not to bioactivity.tsv (D15)"),
]
omitted = pd.DataFrame(OMITTED, columns=["what", "records", "why"])
print("total records not reaching the database:", int(omitted.records.sum()))
omitted

total records not reaching the database: 16637


,what,records,why
0,in vivo records,1265,"organism and dose, 1215 with a dose (D5)"
1,references,1816,"1098 DOIs, 580 PubMed ids, 138 raw urls (D4)"
2,class values,1372,56 distinct (D3)
3,subClass values,1321,329 distinct (D3)
4,control compound names,418,"names only, no structures (D2)"
5,ratings,3741,"3 fields per probe, 0 for all 260 unsuitable (D2)"
6,pains / toxicophore,2494,91 and 247 Yes; kept for the 260 unsuitable in this entry (D2)
7,"published_date, canSAR_ID, URL",3741,kept for the 260 unsuitable in this entry (D2)
8,± error,284,no numeric column (D8)
9,range high,103,no numeric column (D8)
